# 06 — Molecular Docking of Predicted EGFR Inhibitors

## Objective

This notebook evaluates the binding potential of 10 candidate EGFR inhibitors identified through machine-learning-guided virtual screening of the ZINC22 compound library.

The candidates were previously prioritized based on predicted pIC50, Lipinski drug-likeness criteria, structural similarity to the EGFR training dataset, and molecular diversity.

Molecular docking will be performed using AutoDock Vina to investigate predicted binding orientations, binding affinities, and protein–ligand interactions within the EGFR kinase binding pocket.

## Workflow

1. **Protein Structure Retrieval:** Obtain and inspect the EGFR crystal structure (PDB: 6LUD).
2. **Receptor Preparation:** Prepare the protein structure for molecular docking.
3. **Ligand Preparation:** Generate and optimize 3D structures of the 10 selected ZINC compounds.
4. **Binding Site Identification:** Define the docking region using the experimentally observed inhibitor-binding pocket.
5. **Docking Validation:** Evaluate the docking protocol using an appropriate reference ligand.
6. **Molecular Docking:** Dock the selected compounds using AutoDock Vina.
7. **Results Analysis:** Compare predicted binding affinities, binding poses, and key protein–ligand interactions.

## Expected Outputs

- Prepared EGFR receptor and ligand structures
- Docking scores for all 10 candidate compounds
- Predicted protein–ligand binding poses
- Visualizations of selected protein–ligand interactions
- Comparison of molecular docking results with machine-learning predictions

**Note:** PDB 6LUD represents a mutant EGFR structure containing L858R/T790M/C797S substitutions and the covalent inhibitor osimertinib. Conventional AutoDock Vina docking evaluates noncovalent interactions; therefore, docking scores will be interpreted as computational estimates rather than experimentally validated binding affinities.

In [2]:
from pathlib import Path
from collections import Counter
import urllib.request

PROJECT_ROOT = Path("..").resolve()

RECEPTOR_DIR = PROJECT_ROOT / "docking" / "input"
RECEPTOR_DIR.mkdir(parents=True, exist_ok=True)

pdb_path = RECEPTOR_DIR / "6LUD.pdb"

if not pdb_path.exists():
    urllib.request.urlretrieve(
        "https://files.rcsb.org/download/6LUD.pdb",
        pdb_path
    )

print("Structure:", pdb_path)
print("File size:", pdb_path.stat().st_size, "bytes")

with open(pdb_path) as f:
    lines = f.readlines()

chains = sorted({
    line[21].strip()
    for line in lines
    if line.startswith("ATOM  ")
})

ligands = Counter(
    (line[17:20].strip(), line[21].strip())
    for line in lines
    if line.startswith("HETATM")
    and line[17:20].strip() not in {"HOH"}
)

print("Protein chains:", chains)
print("Non-water hetero residues:", ligands)

Structure: /Users/gracescanlan/Desktop/egfr-ml-drug-discovery/docking/input/6LUD.pdb
File size: 242109 bytes
Protein chains: ['A']
Non-water hetero residues: Counter({('YY3', 'A'): 37})


## 2. Receptor and Reference Ligand Extraction

The EGFR crystal structure (PDB: 6LUD) was inspected to identify its protein chain and co-crystallized ligand. Protein atoms from chain A and the reference ligand (YY3) were extracted into separate PDB files for subsequent receptor preparation and binding-site analysis.

In [3]:
# Define output paths
protein_path = RECEPTOR_DIR / "egfr_protein.pdb"
ligand_path = RECEPTOR_DIR / "egfr_reference_YY3.pdb"

# Extract protein atoms from chain A
protein_lines = [
    line for line in lines
    if line.startswith("ATOM  ") and line[21] == "A"
]

# Extract co-crystallized ligand YY3
ligand_lines = [
    line for line in lines
    if line.startswith("HETATM")
    and line[17:20].strip() == "YY3"
    and line[21] == "A"
]

# Save extracted structures
with open(protein_path, "w") as f:
    f.writelines(protein_lines)
    f.write("END\n")

with open(ligand_path, "w") as f:
    f.writelines(ligand_lines)
    f.write("END\n")

print("Protein atoms:", len(protein_lines))
print("Ligand atoms:", len(ligand_lines))

print("\nSaved protein:", protein_path)
print("Saved reference ligand:", ligand_path)

Protein atoms: 2388
Ligand atoms: 37

Saved protein: /Users/gracescanlan/Desktop/egfr-ml-drug-discovery/docking/input/egfr_protein.pdb
Saved reference ligand: /Users/gracescanlan/Desktop/egfr-ml-drug-discovery/docking/input/egfr_reference_YY3.pdb


## 3. Receptor Structure Inspection

The extracted EGFR protein structure was examined to identify its residue composition, alternate atomic conformations, and potential structural issues before receptor preparation. This inspection was performed to assess the suitability of the crystallographic structure for molecular docking.

In [4]:
# %pip install biopython

In [5]:
from Bio.PDB import PDBParser
from collections import Counter

parser = PDBParser(QUIET=True)

structure = parser.get_structure(
    "EGFR",
    str(protein_path)
)

model = structure[0]
chain = model["A"]

residues = list(chain.get_residues())
atoms = list(chain.get_atoms())

print("Protein chain:", chain.id)
print("Number of residues:", len(residues))
print("Number of atoms:", len(atoms))

# Residue numbering
residue_numbers = [
    residue.id[1]
    for residue in residues
]

print(
    "Residue range:",
    min(residue_numbers),
    "-",
    max(residue_numbers)
)

# Identify gaps in residue numbering
missing_positions = sorted(
    set(range(min(residue_numbers), max(residue_numbers) + 1))
    - set(residue_numbers)
)

print("Missing residue positions:", missing_positions)

# Identify alternate conformations
alternate_atoms = [
    atom
    for atom in atoms
    if atom.is_disordered()
]

print("Atoms with alternate conformations:", len(alternate_atoms))

# Amino acid composition
residue_counts = Counter(
    residue.resname
    for residue in residues
)

print("\nResidue composition:")
print(residue_counts)

Protein chain: A
Number of residues: 310
Number of atoms: 2388
Residue range: 696 - 1018
Missing residue positions: [748, 749, 750, 751, 991, 992, 993, 994, 995, 996, 997, 998, 999]
Atoms with alternate conformations: 0

Residue composition:
Counter({'LEU': 32, 'ILE': 25, 'GLU': 23, 'VAL': 23, 'ALA': 22, 'LYS': 22, 'ASP': 20, 'GLY': 19, 'PRO': 16, 'ARG': 15, 'SER': 15, 'MET': 13, 'TYR': 12, 'GLN': 10, 'THR': 10, 'HIS': 8, 'ASN': 7, 'PHE': 7, 'TRP': 6, 'CYS': 5})


## 4. Binding-Site Inspection

The EGFR ligand-binding pocket was characterized using the coordinates of the co-crystallized reference ligand (YY3). Protein residues within 5 Å of the ligand were identified to determine the local binding-site environment and assess whether unresolved regions could affect subsequent molecular docking.

In [6]:
import numpy as np
from Bio.PDB import PDBParser
from scipy.spatial.distance import cdist
import pandas as pd

parser = PDBParser(QUIET=True)

protein = parser.get_structure("EGFR", str(protein_path))
ligand = parser.get_structure("YY3", str(ligand_path))

# Extract ligand heavy-atom coordinates
ligand_coords = np.array([
    atom.coord
    for atom in ligand.get_atoms()
    if atom.element != "H"
])

# Calculate geometric center of the ligand
ligand_center = ligand_coords.mean(axis=0)

print("Ligand center (Å):", np.round(ligand_center, 3))

# Identify protein residues within 5 Å
binding_residues = []

for residue in protein.get_residues():

    residue_coords = np.array([
        atom.coord
        for atom in residue.get_atoms()
        if atom.element != "H"
    ])

    if len(residue_coords) == 0:
        continue

    distances = cdist(residue_coords, ligand_coords)

    min_distance = distances.min()

    if min_distance <= 5.0:
        binding_residues.append({
            "Residue": residue.resname,
            "Position": residue.id[1],
            "Min Distance (Å)": round(float(min_distance), 2)
        })

binding_df = pd.DataFrame(binding_residues)

binding_df = binding_df.sort_values("Position")

display(binding_df)

print("\nBinding-site residues:", len(binding_df))

Ligand center (Å): [-4.9731e+01 -3.9000e-02 -1.8220e+01]


,Residue,Position,Min Distance (Å)
0,LEU,718,3.60
1,GLY,719,4.08
2,PHE,723,3.58
3,VAL,726,3.41
4,LYS,728,4.68
5,ALA,743,3.55
6,LYS,745,4.11
7,MET,790,3.98
8,GLN,791,3.43
9,LEU,792,3.56



Binding-site residues: 19


## 5. Docking Grid Definition

The molecular docking search space was defined using the coordinates of the co-crystallized ligand YY3. The geometric center of the ligand was used as the docking-box center, and the box dimensions were calculated from the ligand's spatial extent with additional padding to accommodate ligand flexibility and structural variation among screened compounds.

In [7]:
import numpy as np
import pandas as pd

# Ligand coordinates calculated in Step 4
ligand_min = ligand_coords.min(axis=0)
ligand_max = ligand_coords.max(axis=0)

# Define docking box
BOX_PADDING = 8.0  # Å, added to each side

box_center = (ligand_min + ligand_max) / 2
box_size = (ligand_max - ligand_min) + 2 * BOX_PADDING

docking_box = pd.DataFrame({
    "Axis": ["X", "Y", "Z"],
    "Center (Å)": np.round(box_center, 3),
    "Size (Å)": np.round(box_size, 3)
})

display(docking_box)

print("\nDocking box dimensions calculated.")

,Axis,Center (Å),Size (Å)
0,X,-49.874001,22.877001
1,Y,0.026000,24.719000
2,Z,-18.135000,29.917999



Docking box dimensions calculated.


## 6. Receptor Preparation

The extracted EGFR receptor will be prepared for molecular docking by assigning appropriate hydrogen atoms, atomic charges, and AutoDock-compatible atom types. The prepared receptor will be exported in PDBQT format for use with AutoDock Vina.

In [8]:
import importlib.util
import shutil
import sys

print("Python executable:", sys.executable)

packages = {
    "RDKit": "rdkit",
    "Meeko": "meeko",
    "AutoDock Vina": "vina",
    "PDBFixer": "pdbfixer",
    "OpenMM": "openmm",
    "Biopython": "Bio"
}

print("\nPython packages:")
for name, module in packages.items():
    available = importlib.util.find_spec(module) is not None
    print(f"{name}: {'Installed' if available else 'Not installed'}")

print("\nCommand-line tools:")
for command in ["mk_prepare_receptor.py", "mk_prepare_ligand.py", "vina", "obabel"]:
    print(f"{command}: {shutil.which(command) or 'Not found'}")

Python executable: /usr/local/Caskroom/miniconda/base/envs/egfr-drug-discovery/bin/python

Python packages:
RDKit: Installed
Meeko: Installed
AutoDock Vina: Installed
PDBFixer: Installed
OpenMM: Installed
Biopython: Installed

Command-line tools:
mk_prepare_receptor.py: /usr/local/Caskroom/miniconda/base/envs/egfr-drug-discovery/bin/mk_prepare_receptor.py
mk_prepare_ligand.py: /usr/local/Caskroom/miniconda/base/envs/egfr-drug-discovery/bin/mk_prepare_ligand.py
vina: /usr/local/Caskroom/miniconda/base/envs/egfr-drug-discovery/bin/vina
obabel: /usr/local/Caskroom/miniconda/base/envs/egfr-drug-discovery/bin/obabel


In [9]:
import importlib.util
import shutil

for name in ["vina", "meeko", "pdbfixer", "openmm"]:
    print(
        name,
        "Installed" if importlib.util.find_spec(name) else "Missing"
    )

for command in ["vina", "mk_prepare_receptor.py", "mk_prepare_ligand.py", "obabel"]:
    print(command, shutil.which(command))

vina Installed
meeko Installed
pdbfixer Installed
openmm Installed
vina /usr/local/Caskroom/miniconda/base/envs/egfr-drug-discovery/bin/vina
mk_prepare_receptor.py /usr/local/Caskroom/miniconda/base/envs/egfr-drug-discovery/bin/mk_prepare_receptor.py
mk_prepare_ligand.py /usr/local/Caskroom/miniconda/base/envs/egfr-drug-discovery/bin/mk_prepare_ligand.py
obabel /usr/local/Caskroom/miniconda/base/envs/egfr-drug-discovery/bin/obabel


## 7. Protein Structure Preparation

The extracted EGFR receptor was examined using PDBFixer to identify missing residues, missing heavy atoms, and nonstandard amino acids. Structural corrections were evaluated before hydrogen addition and conversion to AutoDock-compatible PDBQT format. The original crystallographic structure was retained for reference.

In [10]:
from pdbfixer import PDBFixer
from pathlib import Path

# Load extracted EGFR receptor
fixer = PDBFixer(filename=str(protein_path))

# Identify structural issues
fixer.findMissingResidues()
fixer.findNonstandardResidues()
fixer.findMissingAtoms()

print("Missing residues:")
print(fixer.missingResidues)

print("\nNonstandard residues:")
print(fixer.nonstandardResidues)

print("\nResidues with missing heavy atoms:")
for residue, atoms in fixer.missingAtoms.items():
    print(
        residue.name,
        residue.id,
        [atom.name for atom in atoms]
    )

Missing residues:
{}

Nonstandard residues:
[]

Residues with missing heavy atoms:
GLU 697 ['CG', 'CD', 'OE1', 'OE2']
LYS 708 ['CE', 'NZ']
LYS 713 ['CE', 'NZ']
LYS 716 ['CD', 'CE', 'NZ']
GLU 734 ['CG', 'CD', 'OE1', 'OE2']
LYS 737 ['CG', 'CD', 'CE', 'NZ']
LYS 754 ['CG', 'CD', 'CE', 'NZ']
LYS 757 ['CG', 'CD', 'CE', 'NZ']
GLU 758 ['CG', 'CD', 'OE1', 'OE2']
LYS 806 ['CG', 'CD', 'CE', 'NZ']
GLU 872 ['CG', 'CD', 'OE1', 'OE2']
LYS 875 ['CG', 'CD', 'CE', 'NZ']
GLU 922 ['CG', 'CD', 'OE1', 'OE2']
LYS 960 ['CE', 'NZ']
ARG 973 ['CZ', 'NH1', 'NH2']
GLN 982 ['CG', 'CD', 'OE1', 'NE2']
GLU 985 ['CG', 'CD', 'OE1', 'OE2']
ARG 986 ['CG', 'CD', 'NE', 'CZ', 'NH1', 'NH2']
HIS 988 ['CG', 'ND1', 'CE1', 'NE2', 'CD2']
ASP 1003 ['CG', 'OD1', 'OD2']
GLU 1004 ['CG', 'CD', 'OE1', 'OE2']
GLU 1005 ['CG', 'CD', 'OE1', 'OE2']


In [11]:
# Summarize missing structural components

print("Missing residue regions:")
for key, residue_names in fixer.missingResidues.items():
    print(key, residue_names)

print("\nNonstandard residues:")
print(fixer.nonstandardResidues)

print("\nResidues with missing heavy atoms:")
print(len(fixer.missingAtoms))

print("\nMissing heavy atoms near binding pocket:")

pocket_positions = set(binding_df["Position"])

for residue, atoms in fixer.missingAtoms.items():
    if int(residue.id) in pocket_positions:
        print(
            residue.name,
            residue.id,
            [atom.name for atom in atoms]
        )

print("\nMissing terminal atoms:")
for residue, atom_names in fixer.missingTerminals.items():
    print(residue.name, residue.id, atom_names)

Missing residue regions:

Nonstandard residues:
[]

Residues with missing heavy atoms:
22

Missing heavy atoms near binding pocket:

Missing terminal atoms:
ILE 1018 ['OXT']


## 8. Receptor Repair and Hydrogen Addition

Missing heavy atoms were reconstructed using PDBFixer while retaining the experimentally resolved protein structure. Hydrogen atoms were subsequently added at physiological pH (7.4). The prepared receptor was saved as a PDB file for subsequent conversion to AutoDock-compatible PDBQT format.

In [12]:
from pdbfixer import PDBFixer
from openmm.app import PDBFile
from pathlib import Path

# Start from the original extracted protein
fixer = PDBFixer(filename=str(protein_path))

fixer.findMissingResidues()

# Do not automatically reconstruct missing residue segments
fixer.missingResidues = {}

fixer.findNonstandardResidues()
fixer.replaceNonstandardResidues()

fixer.findMissingAtoms()

# Reconstruct missing heavy atoms
fixer.addMissingAtoms()

# Add hydrogens at physiological pH
fixer.addMissingHydrogens(7.4)

# Save prepared receptor
prepared_path = (
    PROJECT_ROOT
    / "docking"
    / "input"
    / "egfr_prepared.pdb"
)

prepared_path.parent.mkdir(
    parents=True,
    exist_ok=True
)

with open(prepared_path, "w") as f:
    PDBFile.writeFile(
        fixer.topology,
        fixer.positions,
        f,
        keepIds=True
    )

print("Prepared receptor saved:", prepared_path)

print(
    "Total atoms:",
    fixer.topology.getNumAtoms()
)

print(
    "Total residues:",
    fixer.topology.getNumResidues()
)

Prepared receptor saved: /Users/gracescanlan/Desktop/egfr-ml-drug-discovery/docking/input/egfr_prepared.pdb
Total atoms: 4981
Total residues: 310


## 9. Prepared Receptor Validation

The prepared EGFR receptor was compared with the original crystallographic structure to assess structural changes introduced during receptor preparation. Root-mean-square deviation (RMSD) was calculated using matched heavy atoms from the original and prepared structures. RMSD was also calculated separately for residues within 5 Å of the co-crystallized ligand to evaluate preservation of the experimentally observed binding pocket.

In [13]:
from Bio.PDB import PDBParser
import numpy as np

parser = PDBParser(QUIET=True)

original = parser.get_structure("original", str(protein_path))
prepared = parser.get_structure("prepared", str(prepared_path))

# Extract heavy atoms, indexed by residue and atom identity
def get_heavy_atoms(structure):
    atoms = {}

    for atom in structure.get_atoms():
        if atom.element == "H":
            continue

        residue = atom.get_parent()
        chain = residue.get_parent()

        key = (
            chain.id,
            residue.id[1],
            residue.id[2],
            atom.name
        )

        atoms[key] = atom.coord

    return atoms

original_atoms = get_heavy_atoms(original)
prepared_atoms = get_heavy_atoms(prepared)

# Match atoms present in both structures
common_keys = sorted(
    set(original_atoms) & set(prepared_atoms)
)

def calculate_rmsd(keys):
    if not keys:
        return np.nan

    differences = np.array([
        original_atoms[key] - prepared_atoms[key]
        for key in keys
    ])

    return np.sqrt(
        np.mean(np.sum(differences**2, axis=1))
    )

# Overall RMSD
overall_rmsd = calculate_rmsd(common_keys)

# Binding-pocket RMSD
pocket_positions = set(binding_df["Position"])

pocket_keys = [
    key for key in common_keys
    if key[1] in pocket_positions
]

pocket_rmsd = calculate_rmsd(pocket_keys)

print("Matched heavy atoms:", len(common_keys))
print("Binding-pocket atoms:", len(pocket_keys))

print(f"\nOverall heavy-atom RMSD: {overall_rmsd:.4f} Å")
print(f"Binding-pocket RMSD: {pocket_rmsd:.4f} Å")

print("\nPrepared receptor residues:",
      len(list(prepared.get_residues())))

Matched heavy atoms: 2388
Binding-pocket atoms: 147

Overall heavy-atom RMSD: 0.0000 Å
Binding-pocket RMSD: 0.0000 Å

Prepared receptor residues: 310


## 10. Receptor Conversion to PDBQT

The prepared EGFR receptor was converted into AutoDock-compatible
PDBQT format for molecular docking. This conversion assigns
appropriate atom types and partial charges while retaining the
prepared protein coordinates.

The resulting receptor structure will be used for docking the
reference ligand and machine learning-prioritized candidate compounds.

In [14]:
import importlib.util
import shutil

print("Meeko:", importlib.util.find_spec("meeko") is not None)
print("AutoDock Vina:", importlib.util.find_spec("vina") is not None)

print("mk_prepare_receptor:",
      shutil.which("mk_prepare_receptor.py"))

print("Vina executable:",
      shutil.which("vina"))

Meeko: True
AutoDock Vina: True
mk_prepare_receptor: /usr/local/Caskroom/miniconda/base/envs/egfr-drug-discovery/bin/mk_prepare_receptor.py
Vina executable: /usr/local/Caskroom/miniconda/base/envs/egfr-drug-discovery/bin/vina


In [15]:
from pathlib import Path
import subprocess
import shutil

# Locate the prepared EGFR receptor
print("Prepared receptor:", prepared_path)
print("File exists:", Path(prepared_path).exists())

# Define output location
receptor_pdbqt = (
    PROJECT_ROOT / "docking" / "input" / "egfr_receptor.pdbqt"
)

receptor_pdbqt.parent.mkdir(parents=True, exist_ok=True)

# Locate Meeko receptor preparation executable
meeko_executable = shutil.which("mk_prepare_receptor.py")

print("\nMeeko executable:", meeko_executable)
print("Output receptor:", receptor_pdbqt)

Prepared receptor: /Users/gracescanlan/Desktop/egfr-ml-drug-discovery/docking/input/egfr_prepared.pdb
File exists: True

Meeko executable: /usr/local/Caskroom/miniconda/base/envs/egfr-drug-discovery/bin/mk_prepare_receptor.py
Output receptor: /Users/gracescanlan/Desktop/egfr-ml-drug-discovery/docking/input/egfr_receptor.pdbqt


In [16]:
# Convert prepared EGFR receptor to PDBQT using Meeko

command = [
    meeko_executable,
    "--read_pdb", str(prepared_path),
    "-o", str(receptor_pdbqt.with_suffix("")),
    "-p"
]

result = subprocess.run(
    command,
    capture_output=True,
    text=True
)

print("Return code:", result.returncode)

print("\nSTDOUT:")
print(result.stdout[-3000:])

print("\nSTDERR:")
print(result.stderr[-3000:])

if result.returncode == 0 and receptor_pdbqt.exists():
    print("\nReceptor PDBQT successfully generated!")
    print("File:", receptor_pdbqt)
else:
    print("\nConversion did not complete.")

Return code: 1

STDOUT:
Template padding will be used.
 gasteiger charges will be read from template file 
 


STDERR:
[RDKit] ERROR:[18:41:51] Explicit valence for atom # 18 N, 5, is greater than permitted
Traceback (most recent call last):
  File "/usr/local/Caskroom/miniconda/base/envs/egfr-drug-discovery/lib/python3.12/site-packages/meeko/polymer.py", line 2534, in _pdb_to_residue_mols
    pdbmol, _, missed_altloc, needed_altloc = _aux_altloc_mol_build(
                                              ^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/Caskroom/miniconda/base/envs/egfr-drug-discovery/lib/python3.12/site-packages/meeko/utils/rdkitutils.py", line 415, in _aux_altloc_mol_build
    _ = Chem.SanitizeMol(pdbmol)
        ^^^^^^^^^^^^^^^^^^^^^^^^
rdkit.Chem.rdchem.AtomValenceException: Explicit valence for atom # 18 N, 5, is greater than permitted

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/usr/local/Caskroom/minicon

In [17]:
from Bio.PDB import PDBParser
from pathlib import Path

parser = PDBParser(QUIET=True)
structure = parser.get_structure("EGFR", str(prepared_path))

# Inspect arginine residues and their atom counts
arginine_residues = []

for model in structure:
    for chain in model:
        for residue in chain:
            if residue.resname == "ARG":
                arginine_residues.append({
                    "Chain": chain.id,
                    "Residue": residue.id[1],
                    "Atoms": len(list(residue.get_atoms())),
                    "Atom names": [
                        atom.name for atom in residue.get_atoms()
                    ]
                })

print("Total arginine residues:", len(arginine_residues))

for residue in arginine_residues[:10]:
    print(residue)

Total arginine residues: 15
{'Chain': 'A', 'Residue': 705, 'Atoms': 24, 'Atom names': ['N', 'H', 'CA', 'HA', 'C', 'O', 'CB', 'HB2', 'HB3', 'CG', 'HG2', 'HG3', 'CD', 'HD2', 'HD3', 'NE', 'HE', 'CZ', 'NH1', 'HH11', 'HH12', 'NH2', 'HH21', 'HH22']}
{'Chain': 'A', 'Residue': 776, 'Atoms': 24, 'Atom names': ['N', 'H', 'CA', 'HA', 'C', 'O', 'CB', 'HB2', 'HB3', 'CG', 'HG2', 'HG3', 'CD', 'HD2', 'HD3', 'NE', 'HE', 'CZ', 'NH1', 'HH11', 'HH12', 'NH2', 'HH21', 'HH22']}
{'Chain': 'A', 'Residue': 803, 'Atoms': 24, 'Atom names': ['N', 'H', 'CA', 'HA', 'C', 'O', 'CB', 'HB2', 'HB3', 'CG', 'HG2', 'HG3', 'CD', 'HD2', 'HD3', 'NE', 'HE', 'CZ', 'NH1', 'HH11', 'HH12', 'NH2', 'HH21', 'HH22']}
{'Chain': 'A', 'Residue': 831, 'Atoms': 24, 'Atom names': ['N', 'H', 'CA', 'HA', 'C', 'O', 'CB', 'HB2', 'HB3', 'CG', 'HG2', 'HG3', 'CD', 'HD2', 'HD3', 'NE', 'HE', 'CZ', 'NH1', 'HH11', 'HH12', 'NH2', 'HH21', 'HH22']}
{'Chain': 'A', 'Residue': 832, 'Atoms': 24, 'Atom names': ['N', 'H', 'CA', 'HA', 'C', 'O', 'CB', 'HB2', 'HB3

In [18]:
import numpy as np
from collections import Counter

for model in structure:
    for chain in model:
        for residue in chain:
            if residue.resname != "ARG":
                continue

            atoms = list(residue.get_atoms())
            names = [atom.name for atom in atoms]

            duplicates = [
                name for name, count in Counter(names).items()
                if count > 1
            ]

            # Check distances between distinct nitrogen atoms
            nitrogens = [
                atom for atom in atoms
                if atom.element == "N"
            ]

            close_pairs = []

            for i in range(len(nitrogens)):
                for j in range(i + 1, len(nitrogens)):
                    distance = nitrogens[i] - nitrogens[j]

                    if distance < 1.5:
                        close_pairs.append((
                            nitrogens[i].name,
                            nitrogens[j].name,
                            round(float(distance), 3)
                        ))

            print(
                f"ARG {chain.id}:{residue.id[1]} | "
                f"Atoms: {len(atoms)} | "
                f"Duplicates: {duplicates} | "
                f"Close N pairs: {close_pairs}"
            )

ARG A:705 | Atoms: 24 | Duplicates: [] | Close N pairs: []
ARG A:776 | Atoms: 24 | Duplicates: [] | Close N pairs: []
ARG A:803 | Atoms: 24 | Duplicates: [] | Close N pairs: []
ARG A:831 | Atoms: 24 | Duplicates: [] | Close N pairs: []
ARG A:832 | Atoms: 24 | Duplicates: [] | Close N pairs: []
ARG A:836 | Atoms: 24 | Duplicates: [] | Close N pairs: []
ARG A:841 | Atoms: 24 | Duplicates: [] | Close N pairs: []
ARG A:858 | Atoms: 24 | Duplicates: [] | Close N pairs: []
ARG A:889 | Atoms: 24 | Duplicates: [] | Close N pairs: []
ARG A:932 | Atoms: 24 | Duplicates: [] | Close N pairs: []
ARG A:958 | Atoms: 24 | Duplicates: [] | Close N pairs: []
ARG A:962 | Atoms: 24 | Duplicates: [] | Close N pairs: []
ARG A:973 | Atoms: 24 | Duplicates: [] | Close N pairs: []
ARG A:977 | Atoms: 24 | Duplicates: [] | Close N pairs: []
ARG A:986 | Atoms: 24 | Duplicates: [] | Close N pairs: []


In [19]:
from Bio.PDB import PDBParser, PDBIO, Select
from pathlib import Path

# Create a separate heavy-atom receptor file
heavy_atom_path = (
    PROJECT_ROOT / "docking" / "input" / "egfr_heavy_atoms.pdb"
)

class HeavyAtomSelect(Select):
    def accept_atom(self, atom):
        return atom.element not in ("H", "D")

parser = PDBParser(QUIET=True)
heavy_structure = parser.get_structure(
    "EGFR_heavy", str(prepared_path)
)

io = PDBIO()
io.set_structure(heavy_structure)
io.save(str(heavy_atom_path), HeavyAtomSelect())

print("Heavy-atom receptor saved:", heavy_atom_path)
print("File exists:", heavy_atom_path.exists())

# Count retained atoms
heavy_atoms = list(
    PDBParser(QUIET=True)
    .get_structure("heavy", str(heavy_atom_path))
    .get_atoms()
)

print("Heavy atoms:", len(heavy_atoms))

Heavy-atom receptor saved: /Users/gracescanlan/Desktop/egfr-ml-drug-discovery/docking/input/egfr_heavy_atoms.pdb
File exists: True
Heavy atoms: 2471


In [20]:
# Attempt receptor conversion using the heavy-atom structure

command = [
    meeko_executable,
    "--read_pdb", str(heavy_atom_path),
    "-o", str(receptor_pdbqt.with_suffix("")),
    "-p"
]

result = subprocess.run(
    command,
    capture_output=True,
    text=True
)

print("Return code:", result.returncode)

print("\nSTDOUT:")
print(result.stdout[-3000:])

print("\nSTDERR:")
print(result.stderr[-3000:])

if result.returncode == 0 and receptor_pdbqt.exists():
    print("\nReceptor PDBQT successfully generated!")
    print("File:", receptor_pdbqt)
    print("File size:", receptor_pdbqt.stat().st_size, "bytes")
else:
    print("\nConversion did not complete.")

Return code: 0

STDOUT:
Template padding will be used.
 gasteiger charges will be read from template file 
 

Files written:
/Users/gracescanlan/Desktop/egfr-ml-drug-discovery/docking/input/egfr_receptor.pdbqt <-- static (i.e., rigid) receptor input file


STDERR:


Receptor PDBQT successfully generated!
File: /Users/gracescanlan/Desktop/egfr-ml-drug-discovery/docking/input/egfr_receptor.pdbqt
File size: 239680 bytes


## 11. Receptor PDBQT Validation

The prepared EGFR receptor was successfully converted into PDBQT
format using Meeko. The resulting structure was inspected to verify
that atomic coordinates, atom types, and partial charges were
retained for molecular docking.

This validation ensures that the receptor file is structurally
consistent before proceeding with ligand preparation and
AutoDock Vina simulations.

In [21]:
# Inspect receptor PDBQT file

with open(receptor_pdbqt, "r") as file:
    lines = file.readlines()

# Extract atom records
atom_lines = [
    line for line in lines
    if line.startswith(("ATOM", "HETATM"))
]

# Extract atom types
atom_types = {}

for line in atom_lines:
    atom_type = line.split()[-1]
    atom_types[atom_type] = atom_types.get(atom_type, 0) + 1

print("Receptor PDBQT validation")
print("-------------------------")
print("File exists:", receptor_pdbqt.exists())
print("Total atom records:", len(atom_lines))
print("Atom types:", atom_types)

print("\nFirst five atom records:")
for line in atom_lines[:5]:
    print(line.strip())

Receptor PDBQT validation
-------------------------
File exists: True
Total atom records: 2996
Atom types: {'C': 1400, 'OA': 451, 'N': 408, 'HD': 525, 'A': 186, 'SA': 18, 'NA': 8}

First five atom records:
ATOM      1  C   GLY A 696     -67.485  -7.797 -50.356  1.00  0.00    +0.239 C
ATOM      2  O   GLY A 696     -68.328  -8.535 -49.830  1.00  0.00    -0.273 OA
ATOM      3  CA  GLY A 696     -67.101  -7.988 -51.818  1.00  0.00    +0.216 C
ATOM      4  N   GLY A 696     -68.077  -7.445 -52.749  1.00  0.00    -0.346 N
ATOM      5  H   GLY A 696     -68.850  -7.293 -53.412  1.00  0.00    +0.164 HD


## 12. Reference Ligand Identification

To validate the molecular docking workflow, the experimentally
resolved EGFR-bound reference ligand will be identified and
prepared for redocking.

The reference ligand will be extracted from the original
crystallographic structure and used to evaluate whether
AutoDock Vina can reproduce its experimentally observed
binding orientation.

Redocking performance will be assessed using heavy-atom
root-mean-square deviation (RMSD).

In [22]:
from pathlib import Path

# Locate PDB files in the docking directory
pdb_files = sorted(
    (PROJECT_ROOT / "docking").rglob("*.pdb")
)

print("Available PDB files:\n")

for path in pdb_files:
    print(path.relative_to(PROJECT_ROOT))

Available PDB files:

docking/input/6LUD.pdb
docking/input/egfr_heavy_atoms.pdb
docking/input/egfr_prepared.pdb
docking/input/egfr_protein.pdb
docking/input/egfr_reference_YY3.pdb


In [23]:
from Bio.PDB import PDBParser
import pandas as pd

# Original EGFR crystal structure
original_pdb_path = (
    PROJECT_ROOT / "docking" / "input" / "6LUD.pdb"
)

parser = PDBParser(QUIET=True)

crystal_structure = parser.get_structure(
    "EGFR_original",
    str(original_pdb_path)
)

# Identify non-protein residues
hetero_residues = []

for residue in crystal_structure.get_residues():
    if residue.id[0].startswith("H_"):
        hetero_residues.append({
            "Residue": residue.resname,
            "Chain": residue.get_parent().id,
            "Residue Number": residue.id[1],
            "Atoms": len(list(residue.get_atoms()))
        })

hetero_df = pd.DataFrame(hetero_residues)

display(hetero_df)

,Residue,Chain,Residue Number,Atoms
0,YY3,A,1101,37


## 13. Reference Ligand Binding-Site Inspection

The original EGFR crystal structure (PDB: 6LUD) contains
the reference ligand YY3 in chain A, residue 1101.

The reference ligand and surrounding protein residues were
examined to characterize the crystallographic binding site
and identify potential covalent interactions.

This assessment will determine whether conventional
noncovalent redocking is appropriate for validation.

In [24]:
import numpy as np
from Bio.PDB import NeighborSearch

# Locate reference ligand
reference_ligand = crystal_structure[0]["A"][("H_YY3", 1101, " ")]

ligand_atoms = list(reference_ligand.get_atoms())

# Protein atoms only
protein_atoms = [
    atom
    for residue in crystal_structure.get_residues()
    if residue.id[0] == " "
    for atom in residue.get_atoms()
    if atom.element not in ("H", "D")
]

# Identify nearby protein residues
neighbor_search = NeighborSearch(protein_atoms)

nearby_residues = set()

for atom in ligand_atoms:
    neighbors = neighbor_search.search(
        atom.coord,
        4.0,
        level="R"
    )
    nearby_residues.update(neighbors)

print("Reference ligand: YY3")
print("Ligand atoms:", len(ligand_atoms))

print("\nProtein residues within 4 Å:")

for residue in sorted(
    nearby_residues,
    key=lambda r: (r.get_parent().id, r.id[1])
):
    print(
        residue.get_parent().id,
        residue.resname,
        residue.id[1]
    )

Reference ligand: YY3
Ligand atoms: 37

Protein residues within 4 Å:
A LEU 718
A PHE 723
A VAL 726
A ALA 743
A MET 790
A GLN 791
A LEU 792
A MET 793
A PRO 794
A GLY 796
A SER 797
A ASP 800
A GLU 804
A LEU 844
A LEU 1001


## 14. Assessment of Potential Covalent Binding

The crystallographic reference ligand YY3 was examined for
proximity to EGFR residue Cys797, a common site of covalent
inhibition.

The minimum distance between Cys797 and YY3 was calculated,
and crystallographic connectivity records were inspected
to assess whether the ligand forms a covalent interaction.

This assessment will inform the selection of an appropriate
reference system for docking validation.

In [25]:
import numpy as np

# Locate Cys797 in the original crystal structure
cys797 = crystal_structure[0]["A"][797]

# Calculate distances between Cys797 and YY3
distances = []

for protein_atom in cys797.get_atoms():
    if protein_atom.element in ("H", "D"):
        continue

    for ligand_atom in ligand_atoms:
        if ligand_atom.element in ("H", "D"):
            continue

        distance = np.linalg.norm(
            protein_atom.coord - ligand_atom.coord
        )

        distances.append({
            "Protein Atom": protein_atom.name,
            "Ligand Atom": ligand_atom.name,
            "Distance (Å)": distance
        })

distance_df = pd.DataFrame(distances).sort_values(
    "Distance (Å)"
)

print("Closest Cys797–YY3 atom pairs:")
display(distance_df.head(10))

# Inspect explicit crystallographic linkage records
with open(original_pdb_path) as file:
    link_records = [
        line.strip()
        for line in file
        if line.startswith(("LINK", "SSBOND"))
    ]

print("\nCrystallographic linkage records:")
print("\n".join(link_records) if link_records else "None found")


Closest Cys797–YY3 atom pairs:


,Protein Atom,Ligand Atom,Distance (Å)
15,N,O,3.126537
163,CB,O,4.068523
52,CA,O,4.149792
14,N,C7,4.155873
202,OG,C9,4.164259
200,OG,O,4.331322
165,CB,C9,4.530244
201,OG,C8,4.911366
17,N,C9,4.946963
199,OG,C7,4.977525



Crystallographic linkage records:
None found


In [26]:
# Inspect the selected Cys797 residue

print("Residue name:", cys797.resname)
print("Chain:", cys797.get_parent().id)
print("Residue number:", cys797.id[1])

print("\nAvailable atoms:")
for atom in cys797.get_atoms():
    print(atom.name, atom.element)

Residue name: SER
Chain: A
Residue number: 797

Available atoms:
N N
CA C
C C
O O
CB C
OG O


In [27]:
# Identify cysteine residues in the original EGFR structure

cysteine_residues = []

for residue in crystal_structure.get_residues():
    if residue.resname == "CYS":
        cysteine_residues.append({
            "Chain": residue.get_parent().id,
            "Residue Number": residue.id[1],
            "Has SG": residue.has_id("SG")
        })

cysteine_df = pd.DataFrame(cysteine_residues)

print("Cysteine residues in the original structure:")
display(cysteine_df)

Cysteine residues in the original structure:


,Chain,Residue Number,Has SG
0,A,775,True
1,A,781,True
2,A,818,True
3,A,939,True
4,A,950,True


In [28]:
import numpy as np
import pandas as pd

# Compare each cysteine sulfur atom with YY3 ligand atoms
sulfur_distances = []

for residue in crystal_structure.get_residues():

    if residue.resname != "CYS" or not residue.has_id("SG"):
        continue

    sg_atom = residue["SG"]

    for ligand_atom in ligand_atoms:

        if ligand_atom.element in ("H", "D"):
            continue

        distance = np.linalg.norm(
            sg_atom.coord - ligand_atom.coord
        )

        sulfur_distances.append({
            "Cysteine": f"{residue.get_parent().id}:{residue.id[1]}",
            "Ligand Atom": ligand_atom.name,
            "Element": ligand_atom.element,
            "Distance (Å)": round(float(distance), 3)
        })

sulfur_df = pd.DataFrame(sulfur_distances)

closest_sulfur = (
    sulfur_df.sort_values("Distance (Å)")
    .groupby("Cysteine", as_index=False)
    .first()
    .sort_values("Distance (Å)")
)

display(closest_sulfur)

,Cysteine,Ligand Atom,Element,Distance (Å)
0,A:775,C16,C,6.113
1,A:781,C20,C,17.108
2,A:818,O,O,19.230
3,A:939,C9,C,23.514
4,A:950,C9,C,24.583


## 16. Molecular Docking Search Space Definition

The molecular docking search space was defined using the experimentally
resolved position of the YY3 reference ligand in the EGFR crystal
structure (PDB: 6LUD).

The geometric center of the ligand was calculated from its heavy-atom
coordinates and used as the center of the AutoDock Vina search box.
This approach directs docking toward the experimentally observed
ligand-binding pocket.

In [29]:
import numpy as np

# Extract reference ligand heavy-atom coordinates
ligand_coords = np.array([
    atom.coord
    for atom in ligand_atoms
    if atom.element not in ("H", "D")
])

# Calculate geometric center
box_center = ligand_coords.mean(axis=0)

# Calculate ligand coordinate ranges
ligand_min = ligand_coords.min(axis=0)
ligand_max = ligand_coords.max(axis=0)

ligand_dimensions = ligand_max - ligand_min

print("Reference ligand: YY3")
print("\nDocking box center (Å):")
print(f"X: {box_center[0]:.3f}")
print(f"Y: {box_center[1]:.3f}")
print(f"Z: {box_center[2]:.3f}")

print("\nReference ligand dimensions (Å):")
print(f"X: {ligand_dimensions[0]:.3f}")
print(f"Y: {ligand_dimensions[1]:.3f}")
print(f"Z: {ligand_dimensions[2]:.3f}")

Reference ligand: YY3

Docking box center (Å):
X: -49.731
Y: -0.039
Z: -18.220

Reference ligand dimensions (Å):
X: 6.877
Y: 8.719
Z: 13.918


## 17. AutoDock Vina Search Box Configuration

The molecular docking search space was centered on the geometric
center of the crystallographic YY3 reference ligand.

An initial docking box measuring 22 × 22 × 24 Å was selected
to encompass the experimentally observed ligand-binding pocket
while allowing space for ligand translation, rotation, and
conformational flexibility.

The same docking region will be used for reference ligand
redocking and subsequent screening of machine learning-prioritized
candidate compounds.

In [30]:
from pathlib import Path

# Define docking box dimensions (Å)
box_size = [22.0, 22.0, 24.0]

# Save docking configuration
config_path = PROJECT_ROOT / "docking" / "input" / "vina_config.txt"

config_path.parent.mkdir(parents=True, exist_ok=True)

config_text = f"""center_x = {box_center[0]:.3f}
center_y = {box_center[1]:.3f}
center_z = {box_center[2]:.3f}

size_x = {box_size[0]:.1f}
size_y = {box_size[1]:.1f}
size_z = {box_size[2]:.1f}

exhaustiveness = 16
num_modes = 10
energy_range = 3
"""

config_path.write_text(config_text)

print("Docking configuration saved:")
print(config_path)

print("\nConfiguration:")
print(config_path.read_text())

Docking configuration saved:
/Users/gracescanlan/Desktop/egfr-ml-drug-discovery/docking/input/vina_config.txt

Configuration:
center_x = -49.731
center_y = -0.039
center_z = -18.220

size_x = 22.0
size_y = 22.0
size_z = 24.0

exhaustiveness = 16
num_modes = 10
energy_range = 3



## 18. Reference Ligand Preparation

The crystallographic reference ligand YY3 was extracted from the
original EGFR structure (PDB ID: 6LUD).

Before molecular docking, the extracted ligand structure was
inspected to confirm the presence of its experimentally resolved
atomic coordinates.

The reference ligand will subsequently be prepared for AutoDock
Vina docking and used to evaluate the ability of the docking
workflow to reproduce its crystallographic binding pose.

In [31]:
from pathlib import Path
from Bio.PDB import PDBParser

# Locate the extracted YY3 reference ligand
reference_ligand_path = (
    PROJECT_ROOT / "docking" / "input" / "egfr_reference_YY3.pdb"
)

print("Reference ligand:", reference_ligand_path)
print("File exists:", reference_ligand_path.exists())

if reference_ligand_path.exists():

    parser = PDBParser(QUIET=True)

    ligand_structure = parser.get_structure(
        "YY3_reference",
        str(reference_ligand_path)
    )

    ligand_atoms = list(ligand_structure.get_atoms())

    print("Total atoms:", len(ligand_atoms))

    print("\nElements:")
    elements = {}

    for atom in ligand_atoms:
        element = atom.element
        elements[element] = elements.get(element, 0) + 1

    print(elements)

    print("\nFirst five atoms:")

    for atom in ligand_atoms[:5]:
        print(atom.name, atom.coord)

else:
    print("Reference ligand file not found.")

Reference ligand: /Users/gracescanlan/Desktop/egfr-ml-drug-discovery/docking/input/egfr_reference_YY3.pdb
File exists: True
Total atoms: 37

Elements:
{'C': 28, 'N': 7, 'O': 2}

First five atoms:
C13 [-48.818  -2.761 -11.176]
N2 [-48.491  -2.14  -12.46 ]
C12 [-47.918  -3.133 -13.366]
C11 [-49.671  -1.503 -13.052]
C10 [-4.9581e+01  1.5000e-02 -1.3068e+01]


## 19. Reference Ligand Chemical Structure Retrieval

The experimentally resolved YY3 ligand contains 37 heavy atoms
(28 carbon, 7 nitrogen, and 2 oxygen atoms).

To prepare the ligand for molecular docking, its chemical
connectivity and bond orders must be established.

The ligand's chemical component definition will be retrieved
from the RCSB Protein Data Bank and used in subsequent
ligand preparation while preserving the crystallographic
coordinates for redocking validation.

In [32]:
import requests
from pathlib import Path

# RCSB Chemical Component Dictionary
ligand_url = "https://files.rcsb.org/ligands/download/YY3.cif"

ligand_cif_path = (
    PROJECT_ROOT / "docking" / "input" / "YY3.cif"
)

response = requests.get(ligand_url, timeout=30)
response.raise_for_status()

ligand_cif_path.write_bytes(response.content)

print("Ligand chemical definition downloaded:")
print(ligand_cif_path)

print("\nFile exists:", ligand_cif_path.exists())
print("File size:", ligand_cif_path.stat().st_size, "bytes")

Ligand chemical definition downloaded:
/Users/gracescanlan/Desktop/egfr-ml-drug-discovery/docking/input/YY3.cif

File exists: True
File size: 14003 bytes


## 20. Reference Ligand Chemical Structure Validation

The YY3 chemical component definition was retrieved from the
RCSB Protein Data Bank.

The chemical structure will be examined to verify its atom
composition, molecular connectivity, and compatibility with
the experimentally resolved ligand.

This validation is necessary before assigning hydrogens,
preparing the ligand for AutoDock Vina, and performing
reference ligand redocking.

In [33]:
from rdkit import Chem
from rdkit.Chem import Descriptors, rdMolDescriptors

# Read the YY3 chemical component definition
ligand_cif_text = ligand_cif_path.read_text()

# Inspect the chemical component metadata
for line in ligand_cif_text.splitlines():
    if line.startswith((
        "_chem_comp.id",
        "_chem_comp.name",
        "_chem_comp.formula",
        "_chem_comp.type"
    )):
        print(line)

# Compare against the experimentally extracted ligand
reference_heavy_atoms = [
    atom for atom in reference_ligand.get_atoms()
    if atom.element not in ("H", "D")
]

print("\nExperimental YY3 ligand:")
print("Heavy atoms:", len(reference_heavy_atoms))

element_counts = {}

for atom in reference_heavy_atoms:
    element_counts[atom.element] = (
        element_counts.get(atom.element, 0) + 1
    )

print("Element composition:", element_counts)

_chem_comp.id                                    YY3 
_chem_comp.name                                  "N-(2-{[2-(dimethylamino)ethyl](methyl)amino}-4-methoxy-5-{[4-(1-methyl-1H-indol-3-yl)pyrimidin-2-yl]amino}phenyl)prop-2-enamide" 
_chem_comp.type                                  NON-POLYMER 
_chem_comp.formula                               "C28 H33 N7 O2" 
_chem_comp.formula_weight                        499.607 

Experimental YY3 ligand:
Heavy atoms: 37
Element composition: {'C': 28, 'N': 7, 'O': 2}


## 21. Reference Ligand Bond Connectivity

The YY3 chemical component definition was confirmed to match the
experimentally resolved ligand's heavy-atom composition.

The RCSB ligand structure will now be retrieved in SDF format
to obtain explicit bond connectivity and bond orders.

This information will subsequently be combined with the
experimentally resolved coordinates to prepare YY3 for
molecular docking and redocking validation.

In [34]:
from rdkit import Chem
import requests

# Download the RCSB chemical component structure
sdf_url = "https://files.rcsb.org/ligands/download/YY3_ideal.sdf"

ligand_sdf_path = (
    PROJECT_ROOT / "docking" / "input" / "YY3_ideal.sdf"
)

response = requests.get(sdf_url, timeout=30)
response.raise_for_status()

ligand_sdf_path.write_bytes(response.content)

# Load the ligand into RDKit
supplier = Chem.SDMolSupplier(
    str(ligand_sdf_path),
    removeHs=False
)

ligand_mol = next(
    (mol for mol in supplier if mol is not None),
    None
)

if ligand_mol is None:
    raise ValueError("Unable to load YY3 SDF structure.")

# Inspect molecular structure
heavy_atom_count = sum(
    atom.GetAtomicNum() > 1
    for atom in ligand_mol.GetAtoms()
)

print("YY3 SDF successfully loaded")
print("Total atoms:", ligand_mol.GetNumAtoms())
print("Heavy atoms:", heavy_atom_count)
print("Bonds:", ligand_mol.GetNumBonds())
print("Conformers:", ligand_mol.GetNumConformers())

assert heavy_atom_count == 37, "Unexpected heavy-atom count"

YY3 SDF successfully loaded
Total atoms: 70
Heavy atoms: 37
Bonds: 73
Conformers: 1


[18:42:02] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.


## 22. Reference Ligand Atom Mapping

The YY3 chemical structure was successfully loaded using RDKit,
with 37 heavy atoms and 73 bonds.

Atom identifiers from the chemical component definition will
be compared with the experimentally resolved YY3 ligand.

This mapping is necessary to transfer the crystallographic
coordinates while preserving the correct molecular connectivity
and bond orders for subsequent docking preparation.

In [35]:
from collections import Counter

# RDKit atom names from the SDF
sdf_heavy_atoms = [
    atom for atom in ligand_mol.GetAtoms()
    if atom.GetAtomicNum() > 1
]

sdf_atom_names = [
    atom.GetProp("_TriposAtomName")
    if atom.HasProp("_TriposAtomName")
    else atom.GetPDBResidueInfo().GetName().strip()
    if atom.GetPDBResidueInfo() is not None
    else None
    for atom in sdf_heavy_atoms
]

# Experimental atom names from the crystal structure
crystal_atom_names = [
    atom.name for atom in reference_heavy_atoms
]

print("SDF heavy atoms:", len(sdf_atom_names))
print("Crystal heavy atoms:", len(crystal_atom_names))

print("\nSDF atom names:")
print(sdf_atom_names)

print("\nCrystal atom names:")
print(crystal_atom_names)

print("\nSDF atom names available:",
      all(name is not None for name in sdf_atom_names))

if all(name is not None for name in sdf_atom_names):
    print("Atom names match:",
          Counter(sdf_atom_names) == Counter(crystal_atom_names))

SDF heavy atoms: 37
Crystal heavy atoms: 37

SDF atom names:
[None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None]

Crystal atom names:
['C13', 'N2', 'C12', 'C11', 'C10', 'N1', 'C28', 'C3', 'C4', 'O1', 'C14', 'C2', 'C1', 'N', 'C7', 'O', 'C8', 'C9', 'C6', 'C5', 'N3', 'C15', 'N5', 'N4', 'C16', 'C17', 'C18', 'C19', 'C26', 'C25', 'C24', 'C23', 'C22', 'C21', 'N6', 'C27', 'C20']

SDF atom names available: False


## 23. Chemical Component Atom Mapping

The YY3 SDF structure contains the expected 37 heavy atoms,
but its atom identifiers were not preserved during RDKit import.

The RCSB chemical component CIF file will be examined to
retrieve the official atom identifiers and establish their
correspondence with the crystallographic ligand.

Atom identities must be verified before transferring
experimental coordinates to the chemically defined structure.

In [36]:
from Bio.PDB.MMCIF2Dict import MMCIF2Dict
from collections import Counter

# Read YY3 chemical component information
cif_data = MMCIF2Dict(str(ligand_cif_path))

# Extract atom identifiers and elements
cif_atom_names = cif_data["_chem_comp_atom.atom_id"]
cif_elements = cif_data["_chem_comp_atom.type_symbol"]

# Retain heavy atoms
cif_heavy_atoms = [
    (name, element)
    for name, element in zip(cif_atom_names, cif_elements)
    if element.upper() not in ("H", "D")
]

cif_heavy_names = [name for name, element in cif_heavy_atoms]

# Compare against the experimental ligand
crystal_heavy_names = [
    atom.name for atom in reference_heavy_atoms
]

print("CIF heavy atoms:", len(cif_heavy_names))
print("Crystal heavy atoms:", len(crystal_heavy_names))

print("\nCIF atom names:")
print(cif_heavy_names)

print("\nCrystal atom names:")
print(crystal_heavy_names)

print("\nAtom names match:",
      Counter(cif_heavy_names) == Counter(crystal_heavy_names))

print("Unique CIF names:",
      len(set(cif_heavy_names)) == len(cif_heavy_names))

print("Unique crystal names:",
      len(set(crystal_heavy_names)) == len(crystal_heavy_names))

# Check that corresponding elements also agree
cif_element_map = dict(cif_heavy_atoms)

element_mismatches = [
    (atom.name, atom.element, cif_element_map.get(atom.name))
    for atom in reference_heavy_atoms
    if atom.element.upper() !=
       str(cif_element_map.get(atom.name, "")).upper()
]

print("\nElement mismatches:", element_mismatches)

CIF heavy atoms: 37
Crystal heavy atoms: 37

CIF atom names:
['C13', 'N2', 'C12', 'C11', 'C10', 'N1', 'C28', 'C3', 'C4', 'O1', 'C14', 'C2', 'C1', 'N', 'C7', 'O', 'C8', 'C9', 'C6', 'C5', 'N3', 'C15', 'N5', 'N4', 'C16', 'C17', 'C18', 'C19', 'C26', 'C25', 'C24', 'C23', 'C22', 'C21', 'N6', 'C27', 'C20']

Crystal atom names:
['C13', 'N2', 'C12', 'C11', 'C10', 'N1', 'C28', 'C3', 'C4', 'O1', 'C14', 'C2', 'C1', 'N', 'C7', 'O', 'C8', 'C9', 'C6', 'C5', 'N3', 'C15', 'N5', 'N4', 'C16', 'C17', 'C18', 'C19', 'C26', 'C25', 'C24', 'C23', 'C22', 'C21', 'N6', 'C27', 'C20']

Atom names match: True
Unique CIF names: True
Unique crystal names: True

Element mismatches: []


## 24. Verification of SDF Atom Ordering

The CIF and crystallographic YY3 structures contain matching
heavy-atom identifiers and elemental compositions.

The atom ordering of the RDKit SDF structure will now be
compared with the RCSB CIF atom definitions.

This verification is required to correctly assign experimental
coordinates to the chemically defined ligand before molecular
docking preparation.

In [37]:
# Extract heavy atoms from the RDKit SDF in their original order
sdf_heavy_atoms = [
    atom for atom in ligand_mol.GetAtoms()
    if atom.GetAtomicNum() > 1
]

sdf_elements = [
    atom.GetSymbol().upper()
    for atom in sdf_heavy_atoms
]

cif_elements_ordered = [
    element.upper()
    for name, element in cif_heavy_atoms
]

# Compare atom ordering
atom_order_matches = (
    sdf_elements == cif_elements_ordered
)

print("SDF heavy atoms:", len(sdf_elements))
print("CIF heavy atoms:", len(cif_elements_ordered))
print("Element order matches:", atom_order_matches)

# Display the comparison
atom_comparison = pd.DataFrame({
    "SDF Index": [
        atom.GetIdx() for atom in sdf_heavy_atoms
    ],
    "CIF Atom Name": cif_heavy_names,
    "SDF Element": sdf_elements,
    "CIF Element": cif_elements_ordered
})

display(atom_comparison.head(15))

SDF heavy atoms: 37
CIF heavy atoms: 37
Element order matches: True


,SDF Index,CIF Atom Name,SDF Element,CIF Element
0,0,C13,C,C
1,1,N2,N,N
2,2,C12,C,C
3,3,C11,C,C
4,4,C10,C,C
5,5,N1,N,N
6,6,C28,C,C
7,7,C3,C,C
8,8,C4,C,C
9,9,O1,O,O


## 25. Validation of Reference Ligand Bond Connectivity

Although the SDF and CIF structures contain the same number of heavy atoms
and matching elemental sequences, these observations alone do not establish
a unique atom correspondence.

Bond connectivity will be compared between the RDKit molecular structure
and the RCSB chemical component definition using the proposed atom mapping.

Successful validation will support the transfer of experimentally observed
ligand coordinates to the chemically defined reference structure.

In [38]:
from Bio.PDB.MMCIF2Dict import MMCIF2Dict

# Load the chemical component definition
cif_data = MMCIF2Dict(str(ligand_cif_path))

# Extract CIF bond information
bond_atom_1 = cif_data["_chem_comp_bond.atom_id_1"]
bond_atom_2 = cif_data["_chem_comp_bond.atom_id_2"]

# Map CIF atom names to RDKit heavy-atom indices
atom_name_to_index = {
    name: atom.GetIdx()
    for name, atom in zip(cif_heavy_names, sdf_heavy_atoms)
}

# Extract heavy-atom bonds from the CIF
cif_bonds = set()

for atom1, atom2 in zip(bond_atom_1, bond_atom_2):
    if atom1 in atom_name_to_index and atom2 in atom_name_to_index:
        bond = frozenset({
            atom_name_to_index[atom1],
            atom_name_to_index[atom2]
        })
        cif_bonds.add(bond)

# Extract heavy-atom bonds from the RDKit SDF
sdf_bonds = set()

for bond in ligand_mol.GetBonds():
    atom1 = bond.GetBeginAtom()
    atom2 = bond.GetEndAtom()

    if atom1.GetAtomicNum() > 1 and atom2.GetAtomicNum() > 1:
        sdf_bonds.add(frozenset({
            atom1.GetIdx(),
            atom2.GetIdx()
        }))

# Compare connectivity
print("CIF heavy-atom bonds:", len(cif_bonds))
print("SDF heavy-atom bonds:", len(sdf_bonds))
print("Bond connectivity matches:", cif_bonds == sdf_bonds)

print("\nBonds only in CIF:", len(cif_bonds - sdf_bonds))
print("Bonds only in SDF:", len(sdf_bonds - cif_bonds))

CIF heavy-atom bonds: 40
SDF heavy-atom bonds: 40
Bond connectivity matches: True

Bonds only in CIF: 0
Bonds only in SDF: 0


## 26. Reconstruction of the Experimental Reference Ligand

The reference ligand YY3 was reconstructed by combining its validated
chemical connectivity with experimentally determined atomic coordinates
from the EGFR crystal structure (PDB: 6LUD).

The SDF and CIF structures were confirmed to contain 37 heavy atoms
and identical heavy-atom bond connectivity. Experimental coordinates
were assigned using the validated atom correspondence.

The reconstructed ligand will be used for subsequent preparation
and molecular redocking validation.

In [39]:
from rdkit import Chem
from rdkit.Geometry import Point3D
import numpy as np

# Copy the chemically defined ligand
reference_mol = Chem.Mol(ligand_mol)

# Map crystallographic atom names to coordinates
crystal_coordinates = {
    atom.name: atom.coord
    for atom in reference_heavy_atoms
}

# Retrieve the existing RDKit conformer
conformer = reference_mol.GetConformer()

# Assign experimental coordinates to heavy atoms
for atom_name, atom_index in atom_name_to_index.items():

    coords = crystal_coordinates[atom_name]

    conformer.SetAtomPosition(
        atom_index,
        Point3D(
            float(coords[0]),
            float(coords[1]),
            float(coords[2])
        )
    )

# Verify reconstructed heavy-atom coordinates
coordinate_differences = []

for atom_name, atom_index in atom_name_to_index.items():

    rdkit_coords = np.array(
        conformer.GetAtomPosition(atom_index)
    )

    crystal_coords = crystal_coordinates[atom_name]

    distance = np.linalg.norm(
        rdkit_coords - crystal_coords
    )

    coordinate_differences.append(distance)

print("Reference ligand: YY3")
print("Heavy atoms mapped:", len(atom_name_to_index))
print(
    "Maximum coordinate difference:",
    max(coordinate_differences),
    "Å"
)

Reference ligand: YY3
Heavy atoms mapped: 37
Maximum coordinate difference: 0.0 Å


## 27. Reference Ligand Hydrogen Reconstruction

Following the reconstruction of the experimental YY3 ligand geometry,
hydrogen atoms were regenerated using RDKit.

Hydrogen positions were optimized with the MMFF94 force field while
keeping all experimentally determined heavy-atom coordinates fixed.

This procedure prepares a geometrically consistent reference ligand
for subsequent conversion to PDBQT and molecular redocking.

In [40]:
from rdkit import Chem
from rdkit.Chem import AllChem
import numpy as np

# Remove existing hydrogens while retaining heavy-atom coordinates
reference_no_h = Chem.RemoveHs(reference_mol)

# Regenerate hydrogen atoms with 3D coordinates
reference_prepared = Chem.AddHs(
    reference_no_h,
    addCoords=True
)

# Store original heavy-atom positions
original_heavy_coords = np.array([
    list(reference_prepared.GetConformer().GetAtomPosition(i))
    for i in range(reference_no_h.GetNumAtoms())
])

# Initialize MMFF94 force field
properties = AllChem.MMFFGetMoleculeProperties(reference_prepared)

if properties is None:
    raise ValueError("MMFF94 parameters unavailable for YY3.")

force_field = AllChem.MMFFGetMoleculeForceField(
    reference_prepared,
    properties
)

if force_field is None:
    raise ValueError("Unable to initialize MMFF94 force field.")

# Fix every heavy atom during optimization
for i in range(reference_no_h.GetNumAtoms()):
    force_field.AddFixedPoint(i)

# Optimize hydrogen positions
status = force_field.Minimize(maxIts=1000)

# Transfer optimized coordinates back to the RDKit conformer
optimized_coords = np.array(force_field.Positions()).reshape(-1, 3)

conformer = reference_prepared.GetConformer()

for i, coords in enumerate(optimized_coords):
    conformer.SetAtomPosition(i, coords)

# Verify that heavy atoms remain unchanged
final_heavy_coords = optimized_coords[:reference_no_h.GetNumAtoms()]

max_heavy_shift = np.linalg.norm(
    final_heavy_coords - original_heavy_coords,
    axis=1
).max()

print("Reference ligand: YY3")
print("Total atoms:", reference_prepared.GetNumAtoms())
print("Heavy atoms:", reference_no_h.GetNumAtoms())
print("Hydrogen atoms:",
      reference_prepared.GetNumAtoms() - reference_no_h.GetNumAtoms())
print("Optimization status:", status)
print("Maximum heavy-atom displacement:", max_heavy_shift, "Å")

Reference ligand: YY3
Total atoms: 70
Heavy atoms: 37
Hydrogen atoms: 33
Optimization status: 0
Maximum heavy-atom displacement: 0.0 Å


## 28. Export of the Prepared Reference Ligand

The reconstructed YY3 reference ligand was exported in SDF format
following hydrogen reconstruction and constrained geometry optimization.

The exported structure preserves the experimentally determined
heavy-atom coordinates from PDB 6LUD while retaining the validated
molecular connectivity and optimized hydrogen positions.

This structure will serve as the reference ligand for subsequent
PDBQT preparation and molecular redocking validation.

In [41]:
from rdkit import Chem

# Define output directory
ligand_output_dir = PROJECT_ROOT / "docking" / "input"
ligand_output_dir.mkdir(parents=True, exist_ok=True)

# Define output file
reference_sdf_path = ligand_output_dir / "YY3_reference_prepared.sdf"

# Assign molecule name
reference_prepared.SetProp("_Name", "YY3_reference_prepared")

# Save the prepared reference ligand
writer = Chem.SDWriter(str(reference_sdf_path))
writer.write(reference_prepared)
writer.close()

# Reload to validate exported structure
supplier = Chem.SDMolSupplier(
    str(reference_sdf_path),
    removeHs=False
)

saved_mol = next(
    (mol for mol in supplier if mol is not None),
    None
)

if saved_mol is None:
    raise ValueError("Failed to reload the prepared YY3 SDF.")

# Verify the saved heavy-atom coordinates
saved_coords = saved_mol.GetConformer().GetPositions()
expected_coords = reference_prepared.GetConformer().GetPositions()

max_export_difference = np.linalg.norm(
    saved_coords[:37] - expected_coords[:37],
    axis=1
).max()

print("Reference ligand saved successfully!")
print("File:", reference_sdf_path)
print("File exists:", reference_sdf_path.exists())
print("Total atoms:", saved_mol.GetNumAtoms())
print("Heavy atoms:", saved_mol.GetNumHeavyAtoms())
print("Maximum export coordinate difference:",
      round(max_export_difference, 5), "Å")

Reference ligand saved successfully!
File: /Users/gracescanlan/Desktop/egfr-ml-drug-discovery/docking/input/YY3_reference_prepared.sdf
File exists: True
Total atoms: 70
Heavy atoms: 37
Maximum export coordinate difference: 0.0 Å


## 29. Reference Ligand PDBQT Preparation

The experimentally reconstructed YY3 reference ligand will be
prepared for molecular docking using Meeko.

Meeko generates the PDBQT representation required by AutoDock Vina,
including atomic charges, docking atom types, and rotatable-bond
definitions.

Before ligand conversion, the Meeko installation and input
structure will be verified.

In [42]:
import importlib.util
import subprocess
import sys

# Check Meeko availability
meeko_available = importlib.util.find_spec("meeko") is not None

print("Meeko installed:", meeko_available)

if meeko_available:
    import meeko
    print("Meeko location:", meeko.__file__)
else:
    print("Meeko must be installed before continuing.")

# Verify the prepared reference ligand
print("\nReference ligand:")
print("File exists:", reference_sdf_path.exists())
print("File:", reference_sdf_path)
print("File size:", reference_sdf_path.stat().st_size, "bytes")

Meeko installed: True
Meeko location: /usr/local/Caskroom/miniconda/base/envs/egfr-drug-discovery/lib/python3.12/site-packages/meeko/__init__.py

Reference ligand:
File exists: True
File: /Users/gracescanlan/Desktop/egfr-ml-drug-discovery/docking/input/YY3_reference_prepared.sdf
File size: 5948 bytes


## 30. Conversion of the Reference Ligand to PDBQT

The reconstructed YY3 reference ligand was converted from SDF to
PDBQT format using Meeko.

The conversion assigns docking-specific atom types, partial charges,
and rotatable-bond definitions required by AutoDock Vina.

The resulting PDBQT file will be used for redocking validation
against the experimentally determined EGFR–YY3 complex.

In [43]:
import subprocess
import sys

# Define output path
reference_pdbqt_path = (
    PROJECT_ROOT / "docking" / "input" / "YY3_reference.pdbqt"
)

# Run Meeko ligand preparation
command = [
    sys.executable,
    "-m",
    "meeko.cli.mk_prepare_ligand",
    "-i", str(reference_sdf_path),
    "-o", str(reference_pdbqt_path)
]

result = subprocess.run(
    command,
    capture_output=True,
    text=True
)

print("Return code:", result.returncode)

print("\nSTDOUT:")
print(result.stdout[-3000:])

print("\nSTDERR:")
print(result.stderr[-3000:])

if result.returncode == 0 and reference_pdbqt_path.exists():
    print("\nReference ligand PDBQT successfully generated!")
    print("File:", reference_pdbqt_path)
    print("File size:", reference_pdbqt_path.stat().st_size, "bytes")
else:
    print("\nPDBQT conversion did not complete.")

Return code: 0

STDOUT:
Input molecules processed: 1, skipped: 0
PDBQT files written: 1
PDBQT files not written due to error: 0
Input molecules with errors: 0


STDERR:


Reference ligand PDBQT successfully generated!
File: /Users/gracescanlan/Desktop/egfr-ml-drug-discovery/docking/input/YY3_reference.pdbqt
File size: 3853 bytes


## 31. Validation of the Prepared Reference Ligand

The YY3 PDBQT file was inspected to verify successful ligand
preparation for AutoDock Vina.

Validation included examination of atom records, partial charges,
AutoDock atom types, and torsional information. Heavy-atom
coordinates were compared against the prepared SDF structure
to ensure that ligand preparation preserved the experimental
binding geometry.

In [44]:
import numpy as np
import pandas as pd

# Read prepared PDBQT
pdbqt_lines = reference_pdbqt_path.read_text().splitlines()

# Extract atom records
atom_lines = [
    line for line in pdbqt_lines
    if line.startswith(("ATOM", "HETATM"))
]

# Extract torsional information
torsion_lines = [
    line for line in pdbqt_lines
    if line.startswith(("ROOT", "BRANCH", "ENDBRANCH", "TORSDOF"))
]

# Build atom information table
pdbqt_atoms = []

for line in atom_lines:
    pdbqt_atoms.append({
        "Atom": line[12:16].strip(),
        "X": float(line[30:38]),
        "Y": float(line[38:46]),
        "Z": float(line[46:54]),
        "Charge": float(line[70:76]),
        "AutoDock Type": line[77:79].strip()
    })

pdbqt_df = pd.DataFrame(pdbqt_atoms)

print("Reference ligand: YY3")
print("Total PDBQT atoms:", len(pdbqt_df))
print("Heavy atoms in PDBQT:",
      (~pdbqt_df["AutoDock Type"].isin(["H", "HD"])).sum())

print("\nAutoDock atom types:")
print(pdbqt_df["AutoDock Type"].value_counts())

print("\nTorsional information:")
for line in torsion_lines[:10]:
    print(line)

print("\nAtom information:")
display(pdbqt_df.head(10))

Reference ligand: YY3
Total PDBQT atoms: 39
Heavy atoms in PDBQT: 37

AutoDock atom types:
AutoDock Type
A     18
C     10
N      4
NA     3
OA     2
HD     2
Name: count, dtype: int64

Torsional information:
ROOT
BRANCH   3   7
BRANCH   7   9
BRANCH   9  10
BRANCH  10  11
ENDBRANCH  10  11
ENDBRANCH   9  10
ENDBRANCH   7   9
ENDBRANCH   3   7
BRANCH   4  14

Atom information:


,Atom,X,Y,Z,Charge,AutoDock Type
0,C,-50.411,2.059,-15.176,0.076,A
1,C,-51.225,2.416,-16.241,0.144,A
2,C,-49.617,0.916,-15.265,0.064,A
3,C,-49.652,0.118,-16.416,0.067,A
4,C,-50.468,0.493,-17.481,0.058,A
5,C,-51.233,1.656,-17.423,0.086,A
6,N,-48.770,0.516,-14.183,-0.372,N
7,C,-47.719,1.459,-13.816,0.143,C
8,C,-49.581,0.015,-13.068,0.131,C
9,C,-49.671,-1.503,-13.052,0.105,C


## 32. Verification of Reference Ligand Coordinates

The heavy-atom coordinates of the prepared YY3 ligand were compared
between the SDF and PDBQT representations.

Because ligand preparation can change atom ordering, coordinate
matching was performed independently of atom indices. The maximum
coordinate displacement was calculated to verify that the
experimental ligand geometry was preserved during conversion.

In [45]:
import numpy as np
from scipy.optimize import linear_sum_assignment
from scipy.spatial.distance import cdist

# Original prepared SDF heavy-atom coordinates
sdf_conf = reference_prepared.GetConformer()

sdf_heavy_coords = np.array([
    list(sdf_conf.GetAtomPosition(atom.GetIdx()))
    for atom in reference_prepared.GetAtoms()
    if atom.GetAtomicNum() > 1
])

# PDBQT heavy-atom coordinates
pdbqt_heavy = pdbqt_df[
    ~pdbqt_df["AutoDock Type"].isin(["H", "HD"])
]

pdbqt_heavy_coords = pdbqt_heavy[
    ["X", "Y", "Z"]
].to_numpy()

print("SDF heavy atoms:", len(sdf_heavy_coords))
print("PDBQT heavy atoms:", len(pdbqt_heavy_coords))

assert len(sdf_heavy_coords) == len(pdbqt_heavy_coords), (
    "Heavy-atom counts do not match."
)

# Calculate pairwise distances
distance_matrix = cdist(
    sdf_heavy_coords,
    pdbqt_heavy_coords
)

# Find a unique minimum-distance atom assignment
sdf_indices, pdbqt_indices = linear_sum_assignment(
    distance_matrix
)

matched_distances = distance_matrix[
    sdf_indices,
    pdbqt_indices
]

print("\nCoordinate comparison:")
print(f"Mean displacement: {matched_distances.mean():.5f} Å")
print(f"Maximum displacement: {matched_distances.max():.5f} Å")

if matched_distances.max() < 0.01:
    print("\nPASS: Heavy-atom coordinates were preserved.")
else:
    print("\nCHECK: Coordinate differences require investigation.")

SDF heavy atoms: 37
PDBQT heavy atoms: 37

Coordinate comparison:
Mean displacement: 0.00000 Å
Maximum displacement: 0.00000 Å

PASS: Heavy-atom coordinates were preserved.


## 33. Verification of AutoDock Vina Availability

AutoDock Vina availability was assessed before initiating reference
ligand redocking. The software installation, version, and prepared
receptor file were checked to confirm that the docking environment
was ready for computational validation.

In [46]:
import shutil
import subprocess
from pathlib import Path

# Locate AutoDock Vina executable
vina_executable = shutil.which("vina")

print("AutoDock Vina executable:", vina_executable)

if vina_executable is not None:
    result = subprocess.run(
        [vina_executable, "--version"],
        capture_output=True,
        text=True
    )

    print("Vina version:", result.stdout.strip())
    print("Return code:", result.returncode)
else:
    print("AutoDock Vina executable not found.")

# Check docking configuration
config_path = PROJECT_ROOT / "docking" / "input" / "vina_config.txt"

print("\nDocking configuration exists:", config_path.exists())

# Identify available receptor PDBQT files
receptor_files = list(
    (PROJECT_ROOT / "docking").rglob("*receptor*.pdbqt")
)

print("\nPrepared receptor files:")

for file in receptor_files:
    print(file)

AutoDock Vina executable: /usr/local/Caskroom/miniconda/base/envs/egfr-drug-discovery/bin/vina
Vina version: AutoDock Vina v1.2.7
Return code: 0

Docking configuration exists: True

Prepared receptor files:
/Users/gracescanlan/Desktop/egfr-ml-drug-discovery/docking/input/egfr_receptor.pdbqt


## 34. Verification of Molecular Docking Inputs

The prepared EGFR receptor, YY3 reference ligand, and AutoDock
Vina configuration were checked before initiating redocking.

The docking search space was evaluated to confirm that the
experimentally observed reference ligand coordinates fall
within the defined binding-site region.

In [47]:
from pathlib import Path
import numpy as np

# Define input files
docking_input = PROJECT_ROOT / "docking" / "input"

receptor_path = docking_input / "egfr_receptor.pdbqt"
ligand_path = reference_pdbqt_path
config_path = docking_input / "vina_config.txt"

# Verify files
for name, path in {
    "EGFR receptor": receptor_path,
    "YY3 ligand": ligand_path,
    "Vina configuration": config_path
}.items():
    print(f"{name}:")
    print(f"  Exists: {path.exists()}")
    print(f"  Path: {path}")

# Parse docking box parameters
config = {}

for line in config_path.read_text().splitlines():
    if "=" in line:
        key, value = line.split("=", 1)
        config[key.strip()] = value.strip()

center = np.array([
    float(config[f"center_{axis}"])
    for axis in "xyz"
])

size = np.array([
    float(config[f"size_{axis}"])
    for axis in "xyz"
])

# Reference ligand heavy-atom coordinates
coords = pdbqt_heavy_coords

box_min = center - size / 2
box_max = center + size / 2

inside_box = np.all(
    (coords >= box_min) & (coords <= box_max),
    axis=1
)

print("\nDocking box center:", center)
print("Docking box dimensions:", size)

print("\nYY3 heavy atoms inside docking box:")
print(f"{inside_box.sum()} / {len(coords)}")

assert receptor_path.exists(), "Receptor PDBQT missing."
assert ligand_path.exists(), "Ligand PDBQT missing."
assert inside_box.all(), "Some YY3 atoms fall outside the docking box."

print("\nPASS: Docking inputs and search space verified.")

EGFR receptor:
  Exists: True
  Path: /Users/gracescanlan/Desktop/egfr-ml-drug-discovery/docking/input/egfr_receptor.pdbqt
YY3 ligand:
  Exists: True
  Path: /Users/gracescanlan/Desktop/egfr-ml-drug-discovery/docking/input/YY3_reference.pdbqt
Vina configuration:
  Exists: True
  Path: /Users/gracescanlan/Desktop/egfr-ml-drug-discovery/docking/input/vina_config.txt

Docking box center: [-4.9731e+01 -3.9000e-02 -1.8220e+01]
Docking box dimensions: [22. 22. 24.]

YY3 heavy atoms inside docking box:
37 / 37

PASS: Docking inputs and search space verified.


## 35. Reference Ligand Redocking with AutoDock Vina

The experimentally observed YY3 ligand was redocked into the
prepared EGFR binding pocket using AutoDock Vina.

Docking was performed using the previously defined search space,
with an exhaustiveness of 16 and a maximum of 10 output poses.

The resulting binding affinities and predicted ligand poses
were retained for subsequent comparison with the experimental
crystal structure.

This conventional docking procedure evaluates noncovalent
interactions and does not explicitly model covalent bond formation.

In [48]:
import subprocess
from pathlib import Path

# Create output directory
redocking_dir = PROJECT_ROOT / "docking" / "results" / "redocking"
redocking_dir.mkdir(parents=True, exist_ok=True)

# Define output files
redocked_path = redocking_dir / "YY3_redocked.pdbqt"
redocking_log = redocking_dir / "YY3_redocking.log"

# Construct AutoDock Vina command
command = [
    vina_executable,
    "--receptor", str(receptor_path),
    "--ligand", str(ligand_path),
    "--config", str(config_path),
    "--out", str(redocked_path)
]

print("Running YY3 reference ligand redocking...\n")

result = subprocess.run(
    command,
    capture_output=True,
    text=True
)

# Save docking output
redocking_log.write_text(
    result.stdout + "\n" + result.stderr
)

print("Return code:", result.returncode)

print("\nAutoDock Vina output:")
print(result.stdout)

if result.stderr:
    print("\nWarnings/errors:")
    print(result.stderr)

print("\nDocked ligand file exists:", redocked_path.exists())

if result.returncode != 0:
    raise RuntimeError("Reference ligand redocking failed.")

print("\nReference ligand redocking completed successfully.")

Running YY3 reference ligand redocking...

Return code: 0

AutoDock Vina output:
AutoDock Vina v1.2.7
#################################################################
# If you used AutoDock Vina in your work, please cite:          #
#                                                               #
# J. Eberhardt, D. Santos-Martins, A. F. Tillack, and S. Forli  #
# AutoDock Vina 1.2.0: New Docking Methods, Expanded Force      #
# Field, and Python Bindings, J. Chem. Inf. Model. (2021)       #
# DOI 10.1021/acs.jcim.1c00203                                  #
#                                                               #
# O. Trott, A. J. Olson,                                        #
# AutoDock Vina: improving the speed and accuracy of docking    #
# with a new scoring function, efficient optimization and       #
# multithreading, J. Comp. Chem. (2010)                         #
# DOI 10.1002/jcc.21334                                         #
#                                       

## 36. Redocking Validation

Heavy-atom RMSD was calculated for each redocked YY3 pose against
the prepared crystal coordinates in `YY3_reference_prepared.sdf`.

Both structures are already in the receptor frame, so the RMSD
was computed in place. The poses were not superimposed. RDKit's
symmetry-aware RMSD was used, which allows equivalent atom
orderings, including symmetric terminal groups, to be matched.
Hydrogens were excluded.

A top-pose heavy-atom RMSD below 2 Å is the criterion used here
for a successful reproduction of the crystallographic pose.
The best redocked pose meets that criterion: the heavy-atom RMSD
is 0.747 Å, and the Vina score is -8.115 kcal/mol.

YY3 contains an acrylamide and may bind covalently to EGFR Cys797.
AutoDock Vina treats the complex as noncovalent, so this redocking
RMSD is only an approximate check that the search box, receptor,
and ligand preparation are reasonable. It does not test covalent
bond formation.


In [1]:
import shutil
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from rdkit import Chem
from rdkit.Chem import rdMolAlign
from meeko import PDBQTMolecule, RDKitMolCreate

PROJECT_ROOT = Path("..").resolve()
SEED = 42

docking_input = PROJECT_ROOT / "docking" / "input"
redocking_dir = PROJECT_ROOT / "docking" / "results" / "redocking"

crystal_sdf = docking_input / "YY3_reference_prepared.sdf"
redocked_pdbqt = redocking_dir / "YY3_redocked.pdbqt"
receptor_pdbqt = docking_input / "egfr_receptor.pdbqt"
config_path = docking_input / "vina_config.txt"

vina_executable = shutil.which("vina")
if vina_executable is None:
    candidate = Path(sys.executable).resolve().parent / "vina"
    if candidate.exists():
        vina_executable = str(candidate)

print("Vina executable:", vina_executable)
print("Crystal ligand:", crystal_sdf.exists())
print("Redocked poses:", redocked_pdbqt.exists())

crystal = Chem.MolFromMolFile(str(crystal_sdf), removeHs=False)
crystal_heavy = Chem.RemoveHs(crystal)

pose_mol = RDKitMolCreate.from_pdbqt_mol(
    PDBQTMolecule.from_file(str(redocked_pdbqt), skip_typing=True)
)[0]
pose_heavy = Chem.RemoveHs(pose_mol)

vina_scores = [
    float(line.split()[3])
    for line in redocked_pdbqt.read_text().splitlines()
    if line.startswith("REMARK VINA RESULT:")
]

print("Crystal heavy atoms:", crystal_heavy.GetNumAtoms())
print("Docked heavy atoms:", pose_heavy.GetNumAtoms())
print("Docked poses:", pose_heavy.GetNumConformers())
print("Vina scores parsed:", len(vina_scores))

assert crystal_heavy.GetNumAtoms() == pose_heavy.GetNumAtoms()
assert pose_heavy.GetNumConformers() == len(vina_scores) == 10

rmsd_rows = []

for pose_index, vina_score in enumerate(vina_scores, start=1):
    # In-place RMSD: coordinates are not aligned.
    rmsd = rdMolAlign.CalcRMS(
        pose_heavy,
        crystal_heavy,
        pose_index - 1,
        0,
    )
    rmsd_rows.append({
        "Pose": pose_index,
        "Vina score (kcal/mol)": vina_score,
        "Heavy-atom RMSD (Å)": rmsd,
    })

rmsd_table = pd.DataFrame(rmsd_rows)
top_rmsd = float(rmsd_table.loc[0, "Heavy-atom RMSD (Å)"])
top_score = float(rmsd_table.loc[0, "Vina score (kcal/mol)"])
top_pose_under_2A = top_rmsd < 2.0

display(rmsd_table.style.format({
    "Vina score (kcal/mol)": "{:.3f}",
    "Heavy-atom RMSD (Å)": "{:.3f}",
}))

print(f"Top pose score: {top_score:.3f} kcal/mol")
print(f"Top pose heavy-atom RMSD: {top_rmsd:.3f} Å")
print("Top pose is under 2 Å:", top_pose_under_2A)

rmsd_path = redocking_dir / "redocking_rmsd.csv"
rmsd_table.to_csv(rmsd_path, index=False)
print("RMSD table saved:", rmsd_path)


Vina executable: /usr/local/Caskroom/miniconda/base/envs/egfr-drug-discovery/bin/vina
Crystal ligand: True
Redocked poses: True
Crystal heavy atoms: 37
Docked heavy atoms: 37
Docked poses: 10
Vina scores parsed: 10


,Pose,Vina score (kcal/mol),Heavy-atom RMSD (Å)
0,1,-8.115,0.747
1,2,-7.380,9.171
2,3,-7.353,2.203
3,4,-7.246,10.375
4,5,-7.217,7.521
5,6,-7.191,8.776
6,7,-7.179,8.489
7,8,-7.156,10.783
8,9,-7.134,8.144
9,10,-7.118,8.844


Top pose score: -8.115 kcal/mol
Top pose heavy-atom RMSD: 0.747 Å
Top pose is under 2 Å: True
RMSD table saved: /Users/gracescanlan/Desktop/egfr-ml-drug-discovery/docking/results/redocking/redocking_rmsd.csv


## 37. Candidate Ligand Preparation

The 10 selected compounds were prepared for docking from the
canonical SMILES in `results/candidates/final_egfr_candidates.csv`.

For each compound, OpenBabel assigned a single protonation state
at pH 7.4. Full enumeration of alternative protomers was not
performed because a protomer-ensemble tool is not installed.
Hydrogens were then added with RDKit. A 3D conformer was embedded
with ETKDGv3 at random seed 42, with chirality enforced, and
minimized with MMFF94. UFF was reserved for molecules that MMFF94
cannot parameterize.

The input chiral centers were checked against the minimized
structure. One SDF and one Meeko PDBQT were written to
`docking/input/ligands/`. Compounds that failed preparation were
recorded with the reason and were not docked.


In [2]:
import random
from pathlib import Path

import numpy as np
import pandas as pd
from rdkit import Chem
from rdkit.Chem import AllChem
from openbabel import openbabel as ob
from meeko import MoleculePreparation, PDBQTWriterLegacy

random.seed(SEED)
np.random.seed(SEED)

candidates_path = (
    PROJECT_ROOT / "results" / "candidates" / "final_egfr_candidates.csv"
)
ligand_dir = docking_input / "ligands"
ligand_dir.mkdir(parents=True, exist_ok=True)

candidates = pd.read_csv(candidates_path)
print("Candidates:", len(candidates))
print(candidates["zincid"].tolist())


def protonate_smiles(smiles, ph=7.4):
    """Return one pH-adjusted SMILES, or None if OpenBabel fails."""
    conversion = ob.OBConversion()
    conversion.SetInAndOutFormats("smi", "smi")
    ob_mol = ob.OBMol()
    if not conversion.ReadString(ob_mol, smiles):
        return None
    ob_mol.AddHydrogens(False, True, ph)
    ob_mol.DeleteHydrogens()
    protonated = conversion.WriteString(ob_mol).strip()
    return protonated or None


def chirality_preserved(reference_smiles, prepared_mol):
    query = Chem.MolFromSmiles(reference_smiles)
    prepared_heavy = Chem.RemoveHs(prepared_mol)
    return prepared_heavy.HasSubstructMatch(query, useChirality=True)


def minimize_conformer(mol):
    properties = AllChem.MMFFGetMoleculeProperties(mol, mmffVariant="MMFF94")
    if properties is None:
        force_field = AllChem.UFFGetMoleculeForceField(mol)
        if force_field is None:
            raise ValueError("Neither MMFF94 nor UFF could be constructed")
        status = force_field.Minimize(maxIts=1000)
        return "UFF", status

    force_field = AllChem.MMFFGetMoleculeForceField(mol, properties)
    status = force_field.Minimize(maxIts=1000)
    return "MMFF94", status


def prepare_candidate(smiles, seed):
    reference = Chem.MolFromSmiles(smiles)
    if reference is None:
        raise ValueError("RDKit could not parse the canonical SMILES")

    notes = []
    protonated = protonate_smiles(smiles)
    mol = Chem.MolFromSmiles(protonated) if protonated else None

    if mol is None:
        mol = Chem.Mol(reference)
        notes.append("OpenBabel pH 7.4 protonation failed; input SMILES was kept")
    elif not mol.HasSubstructMatch(reference, useChirality=True):
        mol = Chem.Mol(reference)
        notes.append(
            "pH 7.4 protonation did not preserve chirality; input SMILES was kept"
        )
    elif protonated != smiles:
        notes.append("pH 7.4 protonation changed the SMILES")
    else:
        notes.append("pH 7.4 protonation left the SMILES unchanged")

    mol = Chem.AddHs(mol)
    params = AllChem.ETKDGv3()
    params.randomSeed = seed
    params.enforceChirality = True
    embed_status = AllChem.EmbedMolecule(mol, params)
    if embed_status != 0:
        raise ValueError(f"ETKDG embedding failed with status {embed_status}")

    force_field_name, minimize_status = minimize_conformer(mol)
    notes.append(f"{force_field_name} minimization status {minimize_status}")

    if not chirality_preserved(smiles, mol):
        raise ValueError("Chirality was not preserved after 3D coordinate generation")

    return mol, "; ".join(notes)


preparations = []
prepared_molecules = {}

for row in candidates.itertuples(index=False):
    zincid = row.zincid
    smiles = row.canonical_smiles
    sdf_path = ligand_dir / f"{zincid}.sdf"
    pdbqt_path = ligand_dir / f"{zincid}.pdbqt"
    record = {
        "zincid": zincid,
        "canonical_smiles": smiles,
        "sdf_path": str(sdf_path.relative_to(PROJECT_ROOT)),
        "pdbqt_path": str(pdbqt_path.relative_to(PROJECT_ROOT)),
        "chirality_preserved": False,
        "status": "failed",
        "notes": "",
    }

    try:
        molecule, notes = prepare_candidate(smiles, SEED)
        molecule.SetProp("_Name", zincid)

        writer = Chem.SDWriter(str(sdf_path))
        writer.write(molecule)
        writer.close()

        preparator = MoleculePreparation()
        setups = preparator.prepare(molecule)
        if not setups:
            raise ValueError("Meeko returned no PDBQT setup")
        pdbqt_string, is_ok, error_message = PDBQTWriterLegacy.write_string(setups[0])
        if not is_ok:
            raise ValueError(f"Meeko PDBQT writing failed: {error_message}")
        pdbqt_path.write_text(pdbqt_string)

        record["chirality_preserved"] = True
        record["status"] = "prepared"
        record["notes"] = notes
        prepared_molecules[zincid] = pdbqt_path
        print(f"Prepared {zincid}")
    except Exception as error:
        record["notes"] = str(error)
        print(f"FAILED {zincid}: {error}")

    preparations.append(record)

preparation_table = pd.DataFrame(preparations)
display(preparation_table)

preparation_table.to_csv(ligand_dir / "preparation_summary.csv", index=False)

prepared_ok = preparation_table[preparation_table["status"] == "prepared"]
failed = preparation_table[preparation_table["status"] != "prepared"]
print("Prepared:", len(prepared_ok))
print("Failed:", len(failed))
if failed.empty:
    print("No compound failed ligand preparation.")
else:
    print("Failures:")
    display(failed[["zincid", "notes"]])

if prepared_ok.empty:
    raise RuntimeError("No candidate ligands were prepared.")
assert bool(prepared_ok["chirality_preserved"].all())


Candidates: 10
['ZINCow00000g35fQ', 'ZINCpr00000m8N6T', 'ZINCnG00000g0nB3', 'ZINCnc00000UlZfA', 'ZINCpc000015MNGn', 'ZINCmF000009qF0V', 'ZINCpJ00000sUCKz', 'ZINCok00000rV1M6', 'ZINCou00000pgX9h', 'ZINCp8000006CY2l']


Prepared ZINCow00000g35fQ
Prepared ZINCpr00000m8N6T


Prepared ZINCnG00000g0nB3
Prepared ZINCnc00000UlZfA


Prepared ZINCpc000015MNGn
Prepared ZINCmF000009qF0V


Prepared ZINCpJ00000sUCKz


Prepared ZINCok00000rV1M6
Prepared ZINCou00000pgX9h


Prepared ZINCp8000006CY2l


,zincid,canonical_smiles,sdf_path,pdbqt_path,chirality_preserved,status,notes
0,ZINCow00000g35fQ,CSCCNC(=O)C[C@H]1CCN(c2nc(C)nc3ccccc23)C1,docking/input/ligands/ZINCow00000g35fQ.sdf,docking/input/ligands/ZINCow00000g35fQ.pdbqt,True,prepared,pH 7.4 protonation left the SMILES unchanged; ...
1,ZINCpr00000m8N6T,CCc1nc([C@H](NCC(C)(C)[C@H](C)O)c2nnnn2CCSC)cs1,docking/input/ligands/ZINCpr00000m8N6T.sdf,docking/input/ligands/ZINCpr00000m8N6T.pdbqt,True,prepared,pH 7.4 protonation changed the SMILES; MMFF94 ...
2,ZINCnG00000g0nB3,Oc1ccc(-c2cnc(Nc3ncnc4[nH]ccc34)s2)c(F)c1,docking/input/ligands/ZINCnG00000g0nB3.sdf,docking/input/ligands/ZINCnG00000g0nB3.pdbqt,True,prepared,pH 7.4 protonation changed the SMILES; MMFF94 ...
3,ZINCnc00000UlZfA,C[C@H](O)[C@H]1CCN1C(=O)[C@@H](C)NC(=O)c1ccc2c...,docking/input/ligands/ZINCnc00000UlZfA.sdf,docking/input/ligands/ZINCnc00000UlZfA.pdbqt,True,prepared,pH 7.4 protonation left the SMILES unchanged; ...
4,ZINCpc000015MNGn,CCOC(=O)NCCNc1cc(N2CC(=O)N(C)[C@@H](C)C2)nc(C)n1,docking/input/ligands/ZINCpc000015MNGn.sdf,docking/input/ligands/ZINCpc000015MNGn.pdbqt,True,prepared,pH 7.4 protonation left the SMILES unchanged; ...
5,ZINCmF000009qF0V,O=C(CCn1ncc2ccccc21)O[C@H]1CC2CCC1CC2,docking/input/ligands/ZINCmF000009qF0V.sdf,docking/input/ligands/ZINCmF000009qF0V.pdbqt,True,prepared,pH 7.4 protonation changed the SMILES; MMFF94 ...
6,ZINCpJ00000sUCKz,CC[C@@]1(C)CCCN(C(=O)Cn2cc(-c3cc(C)ccc3Cl)nn2)C1,docking/input/ligands/ZINCpJ00000sUCKz.sdf,docking/input/ligands/ZINCpJ00000sUCKz.pdbqt,True,prepared,pH 7.4 protonation changed the SMILES; MMFF94 ...
7,ZINCok00000rV1M6,CCN1CCN(C[C@@H](C)Nc2nc(C)nc3c(C)nn(C)c23)CC1,docking/input/ligands/ZINCok00000rV1M6.sdf,docking/input/ligands/ZINCok00000rV1M6.pdbqt,True,prepared,pH 7.4 protonation changed the SMILES; MMFF94 ...
8,ZINCou00000pgX9h,COC(=O)[C@@]1(C)CCCCCN1C(=O)c1ccn2ncc(Br)c2n1,docking/input/ligands/ZINCou00000pgX9h.sdf,docking/input/ligands/ZINCou00000pgX9h.pdbqt,True,prepared,pH 7.4 protonation left the SMILES unchanged; ...
9,ZINCp8000006CY2l,COC[C@@H](OC)C(=O)NCC[C@H]1CCCCN1c1nc(C)nc(=N)...,docking/input/ligands/ZINCp8000006CY2l.sdf,docking/input/ligands/ZINCp8000006CY2l.pdbqt,True,prepared,pH 7.4 protonation left the SMILES unchanged; ...


Prepared: 10
Failed: 0
No compound failed ligand preparation.


## 38. Candidate Docking

Each prepared candidate was docked with AutoDock Vina into the
same EGFR receptor and search box used for YY3 redocking.
Exhaustiveness was 16 and 10 poses were requested. The random
seed was fixed at 42.

Poses and logs were written to `docking/results/candidates/`.
The best score for each compound was combined with the machine
learning columns from candidate prioritization. The YY3 redocking
score was added as a reference row. This step does not repeat
the YY3 docking run.


In [3]:
import subprocess
import time

results_dir = PROJECT_ROOT / "docking" / "results" / "candidates"
results_dir.mkdir(parents=True, exist_ok=True)

config = {}
for line in config_path.read_text().splitlines():
    if "=" in line:
        key, value = line.split("=", 1)
        config[key.strip()] = value.strip()

print("Exhaustiveness:", config.get("exhaustiveness"))
print("Modes:", config.get("num_modes"))
print("Seed:", SEED)
print("Compounds to dock:", len(prepared_molecules))

docking_rows = []

for row in candidates.itertuples(index=False):
    zincid = row.zincid
    ligand_pdbqt = prepared_molecules.get(zincid)
    if ligand_pdbqt is None:
        print(f"Skipping {zincid}: ligand preparation failed")
        continue

    pose_path = results_dir / f"{zincid}.pdbqt"
    log_path = results_dir / f"{zincid}.log"
    command = [
        vina_executable,
        "--receptor", str(receptor_pdbqt),
        "--ligand", str(ligand_pdbqt),
        "--config", str(config_path),
        "--seed", str(SEED),
        "--out", str(pose_path),
    ]

    print(f"Docking {zincid}...")
    start = time.time()
    completed = subprocess.run(
        command,
        capture_output=True,
        text=True,
        timeout=1800,
    )
    log_path.write_text(completed.stdout + "\n" + completed.stderr)
    elapsed = time.time() - start
    print(f"  return code {completed.returncode} in {elapsed:.1f} s")

    if completed.returncode != 0 or not pose_path.exists():
        print(completed.stderr[-1000:])
        raise RuntimeError(f"Docking failed for {zincid}")

    vina_score = None
    for line in pose_path.read_text().splitlines():
        if line.startswith("REMARK VINA RESULT:"):
            vina_score = float(line.split()[3])
            break
    if vina_score is None:
        raise RuntimeError(f"No Vina score was written for {zincid}")

    docking_rows.append({
        "zincid": zincid,
        "canonical_smiles": row.canonical_smiles,
        "vina_score": vina_score,
        "predicted_pIC50": row.predicted_pIC50,
        "rank": row.rank,
        "combined_score": row.combined_score,
        "max_training_similarity": row.max_training_similarity,
    })
    print(f"  best score {vina_score:.3f} kcal/mol")

yy3_smiles = Chem.MolToSmiles(crystal_heavy, isomericSmiles=True)
score_table = pd.DataFrame(docking_rows)
reference_row = pd.DataFrame([{
    "zincid": "YY3",
    "canonical_smiles": yy3_smiles,
    "vina_score": top_score,
    "predicted_pIC50": np.nan,
    "rank": np.nan,
    "combined_score": np.nan,
    "max_training_similarity": np.nan,
}])
score_table = pd.concat([reference_row, score_table], ignore_index=True)

score_path = PROJECT_ROOT / "docking" / "results" / "candidate_docking_scores.csv"
score_table.to_csv(score_path, index=False)

display(score_table.style.format({
    "vina_score": "{:.3f}",
    "predicted_pIC50": "{:.3f}",
    "combined_score": "{:.3f}",
    "max_training_similarity": "{:.3f}",
}, na_rep=""))

print("Scores saved:", score_path)
print("YY3 reference score:", round(top_score, 3))


Exhaustiveness: 16
Modes: 10
Seed: 42
Compounds to dock: 10
Docking ZINCow00000g35fQ...


  return code 0 in 35.0 s
  best score -6.663 kcal/mol
Docking ZINCpr00000m8N6T...


  return code 0 in 80.1 s
  best score -5.554 kcal/mol
Docking ZINCnG00000g0nB3...


  return code 0 in 21.7 s
  best score -7.761 kcal/mol
Docking ZINCnc00000UlZfA...


  return code 0 in 31.8 s
  best score -7.257 kcal/mol
Docking ZINCpc000015MNGn...


...

  return code 0 in 23.4 s
  best score -6.793 kcal/mol
Docking ZINCou00000pgX9h...


  return code 0 in 66.8 s
  best score -5.163 kcal/mol
Docking ZINCp8000006CY2l...


  return code 0 in 52.5 s
  best score -6.105 kcal/mol


,zincid,canonical_smiles,vina_score,predicted_pIC50,rank,combined_score,max_training_similarity
0,YY3,C=CC(=O)Nc1cc(Nc2nccc(-c3cn(C)c4ccccc34)n2)c(OC)cc1N(C)CCN(C)C,-8.115,,,,
1,ZINCow00000g35fQ,CSCCNC(=O)C[C@H]1CCN(c2nc(C)nc3ccccc23)C1,-6.663,8.312,1.000000,0.910,0.324
2,ZINCpr00000m8N6T,CCc1nc([C@H](NCC(C)(C)[C@H](C)O)c2nnnn2CCSC)cs1,-5.554,8.201,2.000000,0.677,0.186
3,ZINCnG00000g0nB3,Oc1ccc(-c2cnc(Nc3ncnc4[nH]ccc34)s2)c(F)c1,-7.761,7.210,29.000000,0.406,0.397
4,ZINCnc00000UlZfA,C[C@H](O)[C@H]1CCN1C(=O)[C@@H](C)NC(=O)c1ccc2c[nH]nc2c1,-7.257,7.369,6.000000,0.334,0.271
5,ZINCpc000015MNGn,CCOC(=O)NCCNc1cc(N2CC(=O)N(C)[C@@H](C)C2)nc(C)n1,-6.541,7.308,16.000000,0.299,0.269
6,ZINCmF000009qF0V,O=C(CCn1ncc2ccccc21)O[C@H]1CC2CCC1CC2,-7.223,7.298,18.000000,0.293,0.268
7,ZINCpJ00000sUCKz,CC[C@@]1(C)CCCN(C(=O)Cn2cc(-c3cc(C)ccc3Cl)nn2)C1,-7.430,7.294,19.000000,0.281,0.260
8,ZINCok00000rV1M6,CCN1CCN(C[C@@H](C)Nc2nc(C)nc3c(C)nn(C)c23)CC1,-6.793,7.329,11.000000,0.272,0.238
9,ZINCou00000pgX9h,COC(=O)[C@@]1(C)CCCCCN1C(=O)c1ccn2ncc(Br)c2n1,-5.163,7.312,15.000000,0.265,0.240


Scores saved: /Users/gracescanlan/Desktop/egfr-ml-drug-discovery/docking/results/candidate_docking_scores.csv
YY3 reference score: -8.115


## 39. Corrected Protonation States

OpenBabel protonation at pH 7.4 was chemically unrealistic for two
candidates. ZINCpr00000m8N6T placed a positive charge on the secondary
amine between the tetrazole and the thiazole. ZINCok00000rV1M6
protonated both piperazine nitrogens, giving a charge of +2.

Those two compounds were prepared again from the original canonical
SMILES. ZINCpr00000m8N6T was kept neutral. ZINCok00000rV1M6 was
prepared both neutral and mono-protonated. In the mono-protonated
form, only the N-ethyl piperazine nitrogen is protonated. That
nitrogen is the more basic alkyl-substituted piperazine nitrogen and
is farther from the aromatic amine than the other piperazine nitrogen.

The neutral form of ZINCpr00000m8N6T and the mono-protonated form of
ZINCok00000rV1M6 are the states used in the docking analysis. The
original OpenBabel poses and scores are retained and are not replaced.
The other eight candidates and the YY3 redocking were not changed.

Embedding, minimization, and PDBQT conversion used the same settings
as the first preparation: ETKDGv3 with random seed 42, enforced
chirality, MMFF94, and Meeko.


In [1]:

import shutil
import subprocess
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
from rdkit import Chem
from rdkit.Chem import AllChem, rdmolops
from meeko import MoleculePreparation, PDBQTWriterLegacy

PROJECT_ROOT = Path("..").resolve()
SEED = 42
np.random.seed(SEED)

docking_input = PROJECT_ROOT / "docking" / "input"
ligand_dir = docking_input / "ligands"
results_dir = PROJECT_ROOT / "docking" / "results" / "candidates"
score_path = PROJECT_ROOT / "docking" / "results" / "candidate_docking_scores.csv"
config_path = docking_input / "vina_config.txt"
receptor_pdbqt = docking_input / "egfr_receptor.pdbqt"
candidates_path = (
    PROJECT_ROOT / "results" / "candidates" / "final_egfr_candidates.csv"
)

vina_executable = shutil.which("vina")
if vina_executable is None:
    candidate = Path(sys.executable).resolve().parent / "vina"
    if candidate.exists():
        vina_executable = str(candidate)

catalog = pd.read_csv(candidates_path).set_index("zincid")


def minimize_conformer(mol):
    properties = AllChem.MMFFGetMoleculeProperties(mol, mmffVariant="MMFF94")
    if properties is None:
        force_field = AllChem.UFFGetMoleculeForceField(mol)
        if force_field is None:
            raise ValueError("Neither MMFF94 nor UFF could be constructed")
        status = force_field.Minimize(maxIts=1000)
        return "UFF", status
    force_field = AllChem.MMFFGetMoleculeForceField(mol, properties)
    status = force_field.Minimize(maxIts=1000)
    return "MMFF94", status


def mono_protonated_piperazine(smiles):
    """Protonate only the N-alkyl piperazine nitrogen farthest from the aromatic amine."""
    mol = Chem.MolFromSmiles(smiles)
    aromatic_amines = [
        atom.GetIdx()
        for atom in mol.GetAtoms()
        if atom.GetSymbol() == "N"
        and atom.GetTotalNumHs() == 1
        and any(neighbor.GetIsAromatic() for neighbor in atom.GetNeighbors())
    ]
    piperazine_nitrogens = [
        atom.GetIdx()
        for atom in mol.GetAtoms()
        if atom.GetSymbol() == "N"
        and not atom.GetIsAromatic()
        and atom.GetDegree() == 3
        and atom.GetTotalNumHs() == 0
    ]
    if len(aromatic_amines) != 1 or len(piperazine_nitrogens) != 2:
        raise ValueError("Piperazine protonation sites were not identified")

    def distance_to_aromatic_amine(atom_index):
        return min(
            len(rdmolops.GetShortestPath(mol, atom_index, amine)) - 1
            for amine in aromatic_amines
        )

    basic_nitrogen = max(piperazine_nitrogens, key=distance_to_aromatic_amine)
    mol.GetAtomWithIdx(basic_nitrogen).SetFormalCharge(1)
    Chem.SanitizeMol(mol)
    if Chem.GetFormalCharge(mol) != 1:
        raise ValueError("Mono-protonation did not produce a charge of +1")
    return mol, basic_nitrogen


def build_state(smiles, state):
    if state == "neutral":
        mol = Chem.MolFromSmiles(smiles)
        detail = "neutral canonical SMILES"
    elif state == "mono":
        mol, basic_nitrogen = mono_protonated_piperazine(smiles)
        detail = (
            "mono-protonated at piperazine nitrogen "
            f"{basic_nitrogen}, charge +1"
        )
    else:
        raise ValueError(state)
    if mol is None:
        raise ValueError("RDKit could not parse the SMILES")
    return mol, detail


def prepare_state(smiles, state, seed):
    reference = Chem.MolFromSmiles(smiles)
    mol, detail = build_state(smiles, state)
    expected_charge = 0 if state == "neutral" else 1
    if Chem.GetFormalCharge(mol) != expected_charge:
        raise ValueError(f"Expected charge {expected_charge}")

    mol = Chem.AddHs(mol)
    params = AllChem.ETKDGv3()
    params.randomSeed = seed
    params.enforceChirality = True
    embed_status = AllChem.EmbedMolecule(mol, params)
    if embed_status != 0:
        raise ValueError(f"ETKDG embedding failed with status {embed_status}")

    force_field_name, minimize_status = minimize_conformer(mol)
    prepared_heavy = Chem.RemoveHs(mol)
    if not prepared_heavy.HasSubstructMatch(reference, useChirality=True):
        raise ValueError("Chirality was not preserved")

    prepared_smiles = Chem.MolToSmiles(prepared_heavy, isomericSmiles=True)
    notes = (
        f"{detail}; {force_field_name} status {minimize_status}; "
        f"charge {Chem.GetFormalCharge(prepared_heavy)}"
    )
    return mol, prepared_smiles, notes


jobs = [
    ("ZINCpr00000m8N6T", "neutral"),
    ("ZINCok00000rV1M6", "neutral"),
    ("ZINCok00000rV1M6", "mono"),
]

prepared_paths = {}
prep_rows = []

for zincid, state in jobs:
    smiles = catalog.loc[zincid, "canonical_smiles"]
    stem = f"{zincid}_{state}"
    sdf_path = ligand_dir / f"{stem}.sdf"
    pdbqt_path = ligand_dir / f"{stem}.pdbqt"
    if sdf_path.exists() or pdbqt_path.exists():
        raise FileExistsError(f"Refusing to overwrite {stem}")

    molecule, prepared_smiles, notes = prepare_state(smiles, state, SEED)
    molecule.SetProp("_Name", stem)
    writer = Chem.SDWriter(str(sdf_path))
    writer.write(molecule)
    writer.close()

    setups = MoleculePreparation().prepare(molecule)
    if not setups:
        raise ValueError(f"Meeko returned no setup for {stem}")
    pdbqt_string, is_ok, error_message = PDBQTWriterLegacy.write_string(setups[0])
    if not is_ok:
        raise ValueError(f"Meeko PDBQT writing failed for {stem}: {error_message}")
    pdbqt_path.write_text(pdbqt_string)

    prepared_paths[(zincid, state)] = pdbqt_path
    prep_rows.append({
        "zincid": zincid,
        "protonation_state": state,
        "prepared_smiles": prepared_smiles,
        "chirality_preserved": True,
        "notes": notes,
    })
    print(f"Prepared {stem}: {notes}")
    print(" ", prepared_smiles)

display(pd.DataFrame(prep_rows))

new_score_rows = []
for zincid, state in jobs:
    stem = f"{zincid}_{state}"
    pose_path = results_dir / f"{stem}.pdbqt"
    log_path = results_dir / f"{stem}.log"
    if pose_path.exists() or log_path.exists():
        raise FileExistsError(f"Refusing to overwrite docking output {stem}")

    command = [
        vina_executable,
        "--receptor", str(receptor_pdbqt),
        "--ligand", str(prepared_paths[(zincid, state)]),
        "--config", str(config_path),
        "--seed", str(SEED),
        "--out", str(pose_path),
    ]
    print(f"Docking {stem}...")
    start = time.time()
    completed = subprocess.run(command, capture_output=True, text=True, timeout=1800)
    log_path.write_text(completed.stdout + "\n" + completed.stderr)
    print(f"  return code {completed.returncode} in {time.time() - start:.1f} s")
    if completed.returncode != 0 or not pose_path.exists():
        print(completed.stderr[-1000:])
        raise RuntimeError(f"Docking failed for {stem}")

    vina_score = None
    for line in pose_path.read_text().splitlines():
        if line.startswith("REMARK VINA RESULT:"):
            vina_score = float(line.split()[3])
            break
    if vina_score is None:
        raise RuntimeError(f"No Vina score was written for {stem}")

    source = catalog.loc[zincid]
    new_score_rows.append({
        "zincid": zincid,
        "canonical_smiles": source["canonical_smiles"],
        "vina_score": vina_score,
        "predicted_pIC50": source["predicted_pIC50"],
        "rank": source["rank"],
        "combined_score": source["combined_score"],
        "max_training_similarity": source["max_training_similarity"],
        "protonation_state": state,
    })
    print(f"  best score {vina_score:.3f} kcal/mol")

scores = pd.read_csv(score_path)
if "protonation_state" not in scores.columns:
    scores["protonation_state"] = "original_obabel"
    scores.loc[scores["zincid"] == "YY3", "protonation_state"] = "reference"

existing = set(zip(scores["zincid"], scores["protonation_state"]))
to_add = [
    row for row in new_score_rows
    if (row["zincid"], row["protonation_state"]) not in existing
]
scores = pd.concat([scores, pd.DataFrame(to_add)], ignore_index=True)
scores.to_csv(score_path, index=False)

display(scores.style.format({
    "vina_score": "{:.3f}",
    "predicted_pIC50": "{:.3f}",
    "combined_score": "{:.3f}",
    "max_training_similarity": "{:.3f}",
}, na_rep=""))
print("Scores updated:", score_path)
print("Analysis forms: ZINCpr00000m8N6T neutral; ZINCok00000rV1M6 mono")


Prepared ZINCpr00000m8N6T_neutral: neutral canonical SMILES; MMFF94 status 0; charge 0
  CCc1nc([C@H](NCC(C)(C)[C@H](C)O)c2nnnn2CCSC)cs1


Prepared ZINCok00000rV1M6_neutral: neutral canonical SMILES; MMFF94 status 0; charge 0
  CCN1CCN(C[C@@H](C)Nc2nc(C)nc3c(C)nn(C)c23)CC1


Prepared ZINCok00000rV1M6_mono: mono-protonated at piperazine nitrogen 2, charge +1; MMFF94 status 0; charge 1
  CC[NH+]1CCN(C[C@@H](C)Nc2nc(C)nc3c(C)nn(C)c23)CC1


,zincid,protonation_state,prepared_smiles,chirality_preserved,notes
0,ZINCpr00000m8N6T,neutral,CCc1nc([C@H](NCC(C)(C)[C@H](C)O)c2nnnn2CCSC)cs1,True,neutral canonical SMILES; MMFF94 status 0; cha...
1,ZINCok00000rV1M6,neutral,CCN1CCN(C[C@@H](C)Nc2nc(C)nc3c(C)nn(C)c23)CC1,True,neutral canonical SMILES; MMFF94 status 0; cha...
2,ZINCok00000rV1M6,mono,CC[NH+]1CCN(C[C@@H](C)Nc2nc(C)nc3c(C)nn(C)c23)CC1,True,"mono-protonated at piperazine nitrogen 2, char..."


Docking ZINCpr00000m8N6T_neutral...


  return code 0 in 53.3 s
  best score -5.604 kcal/mol
Docking ZINCok00000rV1M6_neutral...


  return code 0 in 22.3 s
  best score -6.681 kcal/mol
Docking ZINCok00000rV1M6_mono...


  return code 0 in 23.9 s
  best score -6.624 kcal/mol


,zincid,canonical_smiles,vina_score,predicted_pIC50,rank,combined_score,max_training_similarity,protonation_state
0,YY3,C=CC(=O)Nc1cc(Nc2nccc(-c3cn(C)c4ccccc34)n2)c(OC)cc1N(C)CCN(C)C,-8.115,,,,,reference
1,ZINCow00000g35fQ,CSCCNC(=O)C[C@H]1CCN(c2nc(C)nc3ccccc23)C1,-6.663,8.312,1.000000,0.910,0.324,original_obabel
2,ZINCpr00000m8N6T,CCc1nc([C@H](NCC(C)(C)[C@H](C)O)c2nnnn2CCSC)cs1,-5.554,8.201,2.000000,0.677,0.186,original_obabel
3,ZINCnG00000g0nB3,Oc1ccc(-c2cnc(Nc3ncnc4[nH]ccc34)s2)c(F)c1,-7.761,7.210,29.000000,0.406,0.397,original_obabel
4,ZINCnc00000UlZfA,C[C@H](O)[C@H]1CCN1C(=O)[C@@H](C)NC(=O)c1ccc2c[nH]nc2c1,-7.257,7.369,6.000000,0.334,0.271,original_obabel
5,ZINCpc000015MNGn,CCOC(=O)NCCNc1cc(N2CC(=O)N(C)[C@@H](C)C2)nc(C)n1,-6.541,7.308,16.000000,0.299,0.269,original_obabel
6,ZINCmF000009qF0V,O=C(CCn1ncc2ccccc21)O[C@H]1CC2CCC1CC2,-7.223,7.298,18.000000,0.293,0.268,original_obabel
7,ZINCpJ00000sUCKz,CC[C@@]1(C)CCCN(C(=O)Cn2cc(-c3cc(C)ccc3Cl)nn2)C1,-7.430,7.294,19.000000,0.281,0.260,original_obabel
8,ZINCok00000rV1M6,CCN1CCN(C[C@@H](C)Nc2nc(C)nc3c(C)nn(C)c23)CC1,-6.793,7.329,11.000000,0.272,0.238,original_obabel
9,ZINCou00000pgX9h,COC(=O)[C@@]1(C)CCCCCN1C(=O)c1ccn2ncc(Br)c2n1,-5.163,7.312,15.000000,0.265,0.240,original_obabel


Scores updated: /Users/gracescanlan/Desktop/egfr-ml-drug-discovery/docking/results/candidate_docking_scores.csv
Analysis forms: ZINCpr00000m8N6T neutral; ZINCok00000rV1M6 mono
